# Introduction

Several tests have been performed to assess
- the overall performance of the scheduler 
- which scheduling policies yield better carbon savings and under which conditions
- impact of configuration and external factors on the carbon saving


## Policies

The tests consider several scheduling policies (strategies) for assigning requests to slots:
- `baseline`: assigns requests as they come with the most accurate flavour, resembling the default behavior.
- `carbonshift`: runs DP solver to find optimal assignments, for both slots and flavours
- `hazard`: probability-driven selection of both slot and flavours. The name comes from the gambling "hazard" game, where two dices are rolled, in this case one for slot and one for flavour. Carbon intensity forecast acts as the probability distribution for slot choice, while each flavour has its own probability, so that on average the threshold is respected and the selection is balanced.
- `swarm/MAB`: uses multi-armed bandit to select slots, balancing exploration and exploitation. Cheapest possible slot is always preferred, respecting the error threshold.
- `swarm/ACO`: uses ant colony optimization for slot selection, each ant representing a set of assignments, with pheromone deposits on each slot. Cheapest possible slot is always preferred, respecting the error threshold.
- Extra policies, added for comparison purposes:
  - `offline/MAB`: uses MAB with full knowledge of the scenario, no online scheduling is performed.
  - `offline/ACO`: uses ACO with full knowledge, no online scheduling, allowing an ant to explore all assignments from start to finish.


## Information on the tests

Tests are conducted over three main **scenarios**:
  - `scen_1500_92`: scenario with (on average) 1500 requests per slot (randomness seed `92`) 
  - `scen_1800_230`: scenario with (on average) 1800 requests per slot (randomness seed `230`)
  - `scen_2000_120`: scenario with (on average) 2000 requests per slot (randomness seed `120`) 

A scenario a .json file comprising the requests to be scheduled (each having id, arrival time and deadline) and the carbon intensity forecast values for each slot.
Each scenario has 72 time slots.
Essentialy it contains all input data for a test run.\\
The distribution of requests over slots follows the input pattern specified by the test. That is, each scenario may have different load waves, depending on the pattern chosen for a given test run.



The maximum **deadline** slack for requests is set to 14 slots, matching the future window for possible assignments, which is also 14 slots.\
Time slots are meant to be considered **hourly**, so that each slot represents one hour.\
Given that the carbon intensity pattern presents peaks and valleys every 12 hours following the light and dark cycles of the day, with 14 hourly slots, the scheduling of requests can take advantage of the whole variation of the pattern, always seeing both a valley and a peak (not two valleys nor two peaks).\
Using 30min slots (as in previous Carbonshift literature) would double the number of slots in the same 14-hour window to still be able to exploit CI variations, making the tests even more computationally intensive. For testing purposes, we can stick to hourly slots.

The moving **window** is `[-12,+14]` plus another 12 slots in the past with decaying weights applied (from 12/13 to 1/13).
The average error must be kept below the threshold (`4.0` unless otherwise specified), both globally and within each moving window.

Three execution **flavours** are considered:
| flavour | error | exec time |
|---------|-------|-----------|
| Accurate | 0.0 | 60s |
| Balanced | 2.5 | 30s |
| Fast  | 5.0 | 10s |

Beam **pruning** with 3500 retained states is applied where `batch_size >= 8`.

**Cap levels** define the multiplying factor applied to CI when exceeding an amount of requests assigned to a time slot. That is, requests assigned beyond the cap level will have their carbon intensity cost multiplied by the corresponding factor.\
*All policies* take the defined cap levels into account when scheduling requests, except for `baseline`.\
Cap levels are defined in the `capacity_tiers` section of each scenario, and are varied only for specific test runs aimed at assessing their impact on scheduling.


#### Time slots
Each scenario has 72 time slots. That is, there are requests arriving in each of the 72 slots.\
However, the scheduler may place requests beyond the 72th slot, so the 72 slots are not a hard cap for assigning requests.\
Requests placed beyond the 72th slot are allowed, but not counted in the metrics. Slots beyond are inherently less loaded, and would not represent typical conditions. On the other hand, ignoring the slots leads to trimming a variable amount of requests for each run, which however, is more or less always the same.

Using the 72 slots as a hard cap led to having the last few slots to be overloaded, crossing the threshold and hence yielding higher cost than expected.


### Metrics

The primary metrics used to evaluate the performance of the scheduling policies are:
- `carb_saving`: measures the reduction in carbon emissions achieved by the scheduling policy.
- `solver_ms`: the average time taken by the solver to compute the schedule, in milliseconds.
- `elapsed_s`: the total elapsed time for the test run, in seconds, accounting also for any overhead outside the solver computation.

There are some execution parameters which have been varied in the experiments, in order to study their impact on the performance metrics. These include:
- `paral`: the number of parallel solver threads used by the scheduler.
- `batch`: the number of requests processed in each batch by the scheduler. 
  - Tests have different configurations, this is in `[1,4,6,8,22]` for comprehensive tests with the Carbonshift DP solver. In case of online policies, it goes up to `[1,8,22,48,96]`. Smaller/specific tests may use fewer batch sizes.
- `threshold`: error threshold applied both globally and within each moving window.
- `cap_levels`: the defined cap levels and their corresponding multiplying factors. In general 2 cap levels are defined: one for moderate exceedance and one triggering strong rebound effect.
```json
      "capacity_tiers": [
        {
          "max_requests": 2000,
          "multiplier": 1.0
        },
        {
          "max_requests": 3000,
          "multiplier": 1.5
        },
        {
          "max_requests": null,
          "multiplier": 5.0
        }
      ]
```
this reads as: "up to 2000 requests, the multiplier is 1.0; up to 3000 requests, the multiplier is 1.5; beyond 3000 requests, the multiplier is 5.0."



## Table of tests


- PO : Phase opposition
- IP : In phase
- PS : Phase shift
- FLAT : Flat arrivals


| Done | index | Pattern | Title | Description | Features | Expected outcome | Outcome |
|---|---|---|---|---|---|---|---|
| [X] | 00 | IP   | Tuning DP | Tune `batch_size` | DP only | Best N=4/6 | Best N=1 due to no fallback occurring |
| [X] | 01 | IP   | Tuning DP | Tune `batch_size`, parallelism | DP only | Parallelism=??? yields top results | ... |
| [X] | 02 | IP   | Tuning DP | Tune `batch_size` with no parallelism| DP only | ... | ... |
| [X] | 03 | IP   | Tuning DP | `batch_size=1` & vary parallelism| DP only | parallelism slows down | ... |
| [X] | 10 | IP   | Tuning PerRequest & Swarm | Tune batch sizes for ACO/MAB/Hazard/singleton_best | Swarm only | ... | ... |
| [X] | 11 | IP   | Tuning ACO | Tune n_ants & n_iter for ACO | ACO only | ... | ... |
| [X] | 12 | IP   | Tuning ACO | Tune n_ants & n_iter for ACO | ACO only | ... | ... |
| [X] | 13 | IP   | Tuning ACO | Tune n_ants & n_iter for ACO | ACO only | ... | ... |
| [X] | 14 | IP   | Paral on PerRequest & Swarm| Check how parallelism impacts on best tuned | PerRequest & Swarm | ... | ... |
| [X] | 20 | IP   | CAP1 | Set cap tiers to 1x & 2x     | DP best / accurate only | ... | ... |
| [X] | 21 | IP   | CAP2 | Set cap tiers to 1.5x & 2x   | DP best / accurate only | ... | ... |
| [X] | 22 | IP   | CAP3 | Set cap tiers to 1x & 1.5x  | DP best / accurate only | ... | ... |
| [X] | 23 | IP   | CAP4 | Set cap tiers to 1x & 1.75x  | DP best / accurate only | ... | ... |
| [X] | 30 | IP   | Full compare | Compare all strategies | Best tuned settings for each strategy | DP should win | ... |
| [X] | 31 | PO   | Full compare | Compare all strategies | Best tuned settings for each strategy | DP should win | ... |
| [X] | 32 | PS   | Full compare | Compare all strategies | Best tuned settings for each strategy | DP should win | ... |
| [X] | 33 | FLAT | Full compare | Compare all strategies | Best tuned settings for each strategy | DP should win | ... |
| [ ] | 40 | IP   | Planet v Profit | Compare all strategies on varying error thresholds | Best tuned settings for each strategy | Threshold is key in saving | ... |
| [ ] | 50 | PO   | PhaseOpposition | Set cap tiers to best factors | DP best / accurate only | ... | ... |
| [ ] | 51 | IP   | InPhase | Set cap tiers to best factors | DP best / accurate only | ... | ... |
| [ ] | 52 | IP   | PhaseShift | Set cap tiers to best factors | DP best / accurate only | ... | ... |


# Imports

In [57]:
from pathlib import Path
import json
import pandas as pd
import re
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex, to_rgb
import seaborn as sns
import sys
import os
from html import escape
from IPython.display import HTML, display as _display

plt.rcParams['figure.dpi'] = 85 
REPO_ROOT = Path().resolve()
RESULTS_DIR = REPO_ROOT / "results"

comparison_metric_sources = {
    "elapsed_s": "elapsed_s",
    "carb_saving": "carb_saving",
    "error": "error",
    "solver_ms": "solver_ms",
    "carb_cost": "carb_cost",
}


# Toggle scenario headings and color bands independently across all tables.
SHOW_SCENARIO_HEADINGS = True
SHOW_SCENARIO_COLORS = True
SCENARIO_COLOR_THEME = "dark"  # "light" or "dark"

_SCENARIO_COLOR_THEMES = {
    "light": {"saturation": 0.5, "lightness": 0.86, "text_color": "#202124"},
    "dark": {"saturation": 0.5, "lightness": 0.36, "text_color": "#f5f5f5"},
}
_SCENARIO_ROW_SHADE_AMOUNT = 0.1
_SCENARIO_COLOR_MAP = {}
_SCENARIO_COLOR_MAP_THEME = None


def _scenario_color_key(scenario_id):
    if pd.isna(scenario_id):
        return None
    return str(scenario_id).replace("_sine_IP_L", "")


def _shade_scenario_color(color, *, toward_white):
    target = (1.0, 1.0, 1.0) if toward_white else (0.0, 0.0, 0.0)
    return to_hex(
        tuple(
            channel + (target_value - channel) * _SCENARIO_ROW_SHADE_AMOUNT
            for channel, target_value in zip(to_rgb(color), target)
        )
    )


def display_by_scenario(data, *, title=None):
    """Display scenario-bearing dataframes as configurable HTML sections."""
    global _SCENARIO_COLOR_MAP_THEME
    if SCENARIO_COLOR_THEME not in _SCENARIO_COLOR_THEMES:
        raise ValueError("SCENARIO_COLOR_THEME must be 'light' or 'dark'")
    theme = _SCENARIO_COLOR_THEMES[SCENARIO_COLOR_THEME]

    if (
        not isinstance(data, pd.DataFrame)
        or "scenario_id" not in data.columns
        or data.empty
    ):
        _display(data)
        return

    if (
        SHOW_SCENARIO_COLORS
        and _SCENARIO_COLOR_MAP_THEME != SCENARIO_COLOR_THEME
    ):
        _SCENARIO_COLOR_MAP.clear()
        _SCENARIO_COLOR_MAP_THEME = SCENARIO_COLOR_THEME

    if SHOW_SCENARIO_COLORS and not _SCENARIO_COLOR_MAP:
        color_source = globals().get("df_all")
        if (
            not isinstance(color_source, pd.DataFrame)
            or "scenario_id" not in color_source.columns
        ):
            color_source = data
        scenario_ids = sorted(
            {
                _scenario_color_key(scenario_id)
                for scenario_id in color_source["scenario_id"].dropna().unique()
            },
            key=str,
        )
        if color_source["scenario_id"].isna().any():
            scenario_ids.append(None)
        palette = sns.husl_palette(
            n_colors=max(len(scenario_ids), 1),
            s=theme["saturation"],
            l=theme["lightness"],
        ).as_hex()
        _SCENARIO_COLOR_MAP.update(
            {
                scenario_id: palette[index]
                for index, scenario_id in enumerate(scenario_ids)
            }
        )

    if title:
        _display(HTML(f"<h2>{escape(str(title))}</h2>"))

    for scenario_id, scenario_rows in data.groupby(
        "scenario_id", sort=True, dropna=False
    ):
        is_missing = pd.isna(scenario_id)
        label = "(missing scenario_id)" if is_missing else str(scenario_id)
        if SHOW_SCENARIO_COLORS:
            color_key = _scenario_color_key(scenario_id)
            if color_key not in _SCENARIO_COLOR_MAP:
                _SCENARIO_COLOR_MAP[color_key] = sns.husl_palette(
                    n_colors=len(_SCENARIO_COLOR_MAP) + 1,
                    s=theme["saturation"],
                    l=theme["lightness"],
                ).as_hex()[-1]
            color = _SCENARIO_COLOR_MAP[color_key]

        if SHOW_SCENARIO_HEADINGS:
            heading_style = (
                f" style='margin:0.8em 0 0.35em;padding:0.25em 0.5em;"
                f"border-left:6px solid {color};background-color:{color};"
                f"color:{theme['text_color']}'"
                if SHOW_SCENARIO_COLORS
                else ""
            )
            _display(HTML(f"<h3{heading_style}>{escape(label)}</h3>"))

        rows = scenario_rows.reset_index(drop=True)
        if not SHOW_SCENARIO_COLORS:
            _display(rows)
            continue

        lighter_color = _shade_scenario_color(color, toward_white=True)
        darker_color = _shade_scenario_color(color, toward_white=False)
        styled_rows = rows.style.set_table_styles(
            [
                {
                    "selector": "thead th",
                    "props": [
                        ("background-color", color),
                        ("color", theme["text_color"]),
                        ("font-weight", "bold"),
                    ],
                },
                {
                    "selector": "tbody tr:nth-child(odd) td",
                    "props": [
                        ("background-color", lighter_color),
                        ("color", theme["text_color"]),
                    ],
                },
                {
                    "selector": "tbody tr:nth-child(even) td",
                    "props": [
                        ("background-color", darker_color),
                        ("color", theme["text_color"]),
                    ],
                },
            ]
        )
        _display(styled_rows)


def compare_metrics_wide(
    data,
    *,
    row_keys,
    comparison_column,
    comparison_values,
    metric_sources,
    filters=None,
    labels=None,
    key_renames=None,
):
    """Pivot selected metrics into ordered columns for each comparison value."""
    filters = filters or {}
    required_columns = [
        *row_keys,
        comparison_column,
        *metric_sources.values(),
        *filters,
    ]
    missing_columns = [column for column in required_columns if column not in data.columns]
    if missing_columns:
        raise ValueError(f"Comparison data is missing columns: {missing_columns}")

    selected = data
    for column, value in filters.items():
        if isinstance(value, (list, tuple, set, frozenset)):
            selected = selected.loc[selected[column].isin(value)]
        else:
            selected = selected.loc[selected[column].eq(value)]
    selected = selected.loc[selected[comparison_column].isin(comparison_values)].copy()
    if selected.empty:
        raise ValueError("No rows found for the requested comparison values and filters")

    duplicate_columns = [*row_keys, comparison_column]
    duplicate_rows = selected.duplicated(duplicate_columns, keep=False)
    if duplicate_rows.any():
        details = selected.loc[duplicate_rows, duplicate_columns].drop_duplicates()
        raise ValueError(
            "Expected one row per comparison key; duplicates found:\n"
            + details.to_string(index=False)
        )

    key_renames = key_renames or {}
    metric_renames = {
        source: metric for metric, source in metric_sources.items()
    }
    selected = selected.rename(columns={**key_renames, **metric_renames})
    output_keys = [key_renames.get(column, column) for column in row_keys]
    metrics = list(metric_sources)
    wide = selected.pivot(
        index=output_keys,
        columns=comparison_column,
        values=metrics,
    ).reindex(
        columns=pd.MultiIndex.from_product([metrics, comparison_values])
    )

    labels = labels or {}
    wide.columns = [
        f"{metric}_{labels.get(value, value)}"
        for metric, value in wide.columns
    ]
    result = wide.reset_index().sort_values(output_keys).reset_index(drop=True)
    comparison_columns = [
        f"{metric}_{labels.get(value, value)}"
        for metric in metrics
        for value in comparison_values
    ]
    return result.loc[:, [*output_keys, *comparison_columns]]

In [58]:
# # Makes table interactive in html
# from itables import init_notebook_mode
# init_notebook_mode(all_interactive=True)


# # Fix table alignment in HTML exported notebook
# from IPython.display import display, HTML

# display(HTML("""
# <style>
#     /* Individua i contenitori delle celle Markdown in VSCode/Jupyter */
#     .rendered_html, 
#     .jp-RenderedHTMLCommon, 
#     div[data-mime-type="text-markdown"] {
#         display: flex !important;
#         flex-direction: column !important;
#         align-items: flex-start !important;
#     }

#     /* Ripristina i margini corretti per la tabella */
#     .rendered_html table, 
#     .jp-RenderedHTMLCommon table,
#     div[data-mime-type="text-markdown"] table {
#         margin-left: 0 !important;
#         margin-right: auto !important;
#         display: table !important;
#     }
# </style>
# """))

In [59]:
# %%capture
# disable output for this cell

# this loads and aggregates the battery test results
from save_results import aggregate_and_save
SORTING_CRITERIA = ["scenario_id", "policy", "batch", "error", "paral", "path"]
df_all = aggregate_and_save(prefix_regex=r"^T", sort_by=SORTING_CRITERIA)

Trovate 45 run directories
 - T00_sine_IP_L_dp_0_1110_1913_20
 - T01_sine_IP_L_dp_0_1114_1157_1
 - T01_sine_IP_L_dp_1_1114_1403_10
 - T01_sine_IP_L_dp_2_1114_1428_20
 - T02_sine_IP_L_dp_0_1113_0004_1
 - T03_sine_IP_L_dp_0_1115_1659_1
 - T03_sine_IP_L_dp_1_1115_1700_5
 - T03_sine_IP_L_dp_2_1115_1701_10
 - T03_sine_IP_L_dp_3_1115_1702_14
 - T03_sine_IP_L_dp_4_1115_1703_20
 - T10_sine_IP_L_online_0_1113_2330_20
 - T10_sine_IP_L_perrequest_0_1113_2330_20
 - T11_sine_IP_L_online_0_1113_1222_20
 - T12_sine_IP_L_online_0_1113_1705_20
 - T13_sine_IP_L_online_0_1113_1740_20
 - T14_sine_IP_L_online_0_1115_1746_1
 - T14_sine_IP_L_online_1_1115_1748_10
 - T14_sine_IP_L_online_2_1115_1748_20
 - T14_sine_IP_L_perrequest_0_1115_1749_1
 - T14_sine_IP_L_perrequest_1_1115_1750_10
 - T14_sine_IP_L_perrequest_2_1115_1751_20
 - T20_sine_IP_L_dp_0_1111_1620_20
 - T21_sine_IP_L_dp_0_1111_1624_20
 - T22_sine_IP_L_dp_0_1111_1622_20
 - T23_sine_IP_L_dp_0_1112_1622_20
 - T30_sine_IP_L_dp_0_1115_2246_20
 - T30_si

In [60]:
import sys

sys.path.insert(0, str(RESULTS_DIR.parent))
from nb_utils import require_output_schema

output_csv = RESULTS_DIR / "aggregated_battery_results.csv"
require_output_schema(output_csv, schema_name="carbonshift.battery-aggregate")
pd.set_option("display.max_columns", None)
df = df_all[df_all["path"].str.contains("^T")]
pd.set_option("display.max_rows", 5)
display_by_scenario(
    df.sort_values(
        by=SORTING_CRITERIA
    )
)



,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T33,scen_1500_92_FLAT,108000,104834,1,dp,72.643588,9.916000,3.998011,55848.164541,315,20,0.014533,20,18.063388,204150.181746,T33_FLAT_dp_0_1117_1110_20/scen_1500_92_FLAT/rust_dp
1,T33,scen_1500_92_FLAT,108000,107697,0,offline_ant_colony,60.825040,10.607000,3.957724,81469.420254,0,20,10557.254543,0,0.000000,207962.996409,T33_FLAT_offline_0_1117_1113_20/scen_1500_92_FLAT/rust_offline/strategy_ant_colony
2,T33,scen_1500_92_FLAT,108000,108000,0,offline_arrival_slot,44.481908,0.036000,2.497315,115698.807550,1,20,2.680226,0,0.000000,208398.386551,T33_FLAT_offline_0_1117_1113_20/scen_1500_92_FLAT/rust_offline/strategy_arrival_slot
3,T33,scen_1500_92_FLAT,108000,107680,0,offline_bandit,58.794528,0.370000,3.977851,85681.437599,0,20,327.365743,0,0.000000,207937.039182,T33_FLAT_offline_0_1117_1113_20/scen_1500_92_FLAT/rust_offline/strategy_bandit
4,T33,scen_1500_92_FLAT,108000,103565,0,offline_hazard,70.934134,0.056000,4.003283,58793.365730,0,20,21.270937,0,0.000000,202276.326323,T33_FLAT_offline_0_1117_1113_20/scen_1500_92_FLAT/rust_offline/strategy_hazard
5,T33,scen_1500_92_FLAT,108000,102751,0,offline_random_slot,46.193834,0.040000,2.500803,108235.172156,1,20,3.452106,0,0.000000,201157.561682,T33_FLAT_offline_0_1117_1113_20/scen_1500_92_FLAT/rust_offline/strategy_random_slot
6,T33,scen_1500_92_FLAT,108000,104836,0,offline_singleton_best,72.875006,0.061000,3.997792,55378.130024,0,20,14.776028,0,0.000000,204159.048021,T33_FLAT_offline_0_1117_1113_20/scen_1500_92_FLAT/rust_offline/strategy_singleton_best
7,T33,scen_1500_92_FLAT,108000,104836,0,offline_singleton_best_accurate,9.835005,0.051000,0.000000,184080.254732,0,20,11.807729,0,0.000000,204159.336503,T33_FLAT_offline_0_1117_1113_20/scen_1500_92_FLAT/rust_offline/strategy_singleton_best_accurate
8,T33,scen_1500_92_FLAT,108000,107157,8,online_ant_colony,47.933929,5.043000,3.910594,107884.876457,0,20,2.353784,20,17.433221,207207.640050,T33_FLAT_online_0_1117_1111_20/scen_1500_92_FLAT/rust_online/online_ant_colony
9,T33,scen_1500_92_FLAT,108000,108000,8,online_arrival_slot,44.420136,3.377000,2.501829,115827.540839,1,20,0.001307,20,11.921592,208398.386551,T33_FLAT_perrequest_0_1117_1111_20/scen_1500_92_FLAT/rust_online/online_arrival_slot


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T20,scen_1500_92_sine_IP_L,105723,95899,1,dp,24.378407,9.929000,0.000000,172580.143867,0,20,0.006329,20,17.952006,228215.429815,T20_sine_IP_L_dp_0_1111_1620_20/scen_1500_92_sine_IP_L/rust_dp
1,T21,scen_1500_92_sine_IP_L,105723,96728,1,dp,24.056880,10.000000,0.000000,161051.007505,0,20,0.006371,20,17.879127,212067.936187,T21_sine_IP_L_dp_0_1111_1624_20/scen_1500_92_sine_IP_L/rust_dp
2,T22,scen_1500_92_sine_IP_L,105723,96052,1,dp,34.246391,9.831000,0.000000,174743.037433,0,20,0.006180,20,17.711413,265754.290321,T22_sine_IP_L_dp_0_1111_1622_20/scen_1500_92_sine_IP_L/rust_dp
3,T23,scen_1500_92_sine_IP_L,105723,95983,1,dp,27.318487,10.467000,0.000000,173196.326556,0,20,0.007110,20,18.083071,238294.882627,T23_sine_IP_L_dp_0_1112_1622_20/scen_1500_92_sine_IP_L/rust_dp
4,T03,scen_1500_92_sine_IP_L,105723,95983,1,dp,78.015795,10.584000,3.996619,52387.884439,204,5,0.014391,5,4.976283,238297.835352,T03_sine_IP_L_dp_1_1115_1700_5/scen_1500_92_sine_IP_L/rust_dp
5,T30,scen_1500_92_sine_IP_L,105723,95985,1,dp,78.000587,10.552000,3.997578,52425.262163,285,20,0.014310,20,18.169708,238303.004233,T30_sine_IP_L_dp_0_1115_2246_20/scen_1500_92_sine_IP_L/rust_dp
6,T03,scen_1500_92_sine_IP_L,105723,95985,1,dp,77.987201,10.547000,3.998125,52457.652617,257,10,0.014496,10,9.768295,238305.236524,T03_sine_IP_L_dp_2_1115_1701_10/scen_1500_92_sine_IP_L/rust_dp
7,T01,scen_1500_92_sine_IP_L,105723,95984,1,dp,78.172579,9.868000,3.998557,52015.133343,114,1,0.012727,1,1.000000,238301.782258,T01_sine_IP_L_dp_0_1114_1157_1/scen_1500_92_sine_IP_L/rust_dp
8,T02,scen_1500_92_sine_IP_L,105723,95984,1,dp,78.172579,8.955000,3.998557,52015.133343,114,1,0.011184,1,1.000000,238301.782258,T02_sine_IP_L_dp_0_1113_0004_1/scen_1500_92_sine_IP_L/rust_dp
9,T03,scen_1500_92_sine_IP_L,105723,95984,1,dp,78.172579,8.976000,3.998557,52015.133343,114,1,0.011279,1,1.000000,238301.782258,T03_sine_IP_L_dp_0_1115_1659_1/scen_1500_92_sine_IP_L/rust_dp


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T31,scen_1500_92_sine_PO_L,105723,103440,1,dp,84.274316,10.539000,3.999130,5588.951992,246,20,0.014427,20,18.531821,35540.279284,T31_sine_PO_L_dp_0_1116_1249_20/scen_1500_92_sine_PO_L/rust_dp
1,T31,scen_1500_92_sine_PO_L,105723,102170,1,dp,69.844077,10.566000,3.999144,59042.482338,239,20,0.014699,20,18.325649,195790.667760,T31_sine_PO_L_dp_0_1116_1625_20/scen_1500_92_sine_PO_L/rust_dp
2,T31,scen_1500_92_sine_PO_L,105723,103828,0,offline_ant_colony,58.442552,10.461000,3.894422,82884.981359,0,20,10420.836449,0,0.000000,199446.755285,T31_sine_PO_L_offline_0_1116_1629_20/scen_1500_92_sine_PO_L/rust_offline/strategy_ant_colony
3,T31,scen_1500_92_sine_PO_L,105723,101812,0,offline_ant_colony,76.810203,10.768000,3.955894,8095.630080,0,20,10723.555774,0,0.000000,34910.310180,T31_sine_PO_L_offline_0_1116_1253_20/scen_1500_92_sine_PO_L/rust_offline/strategy_ant_colony
4,T31,scen_1500_92_sine_PO_L,105723,105723,0,offline_arrival_slot,44.076315,0.038000,2.498037,21125.961742,1,20,2.822858,0,0.000000,37776.412126,T31_sine_PO_L_offline_0_1116_1253_20/scen_1500_92_sine_PO_L/rust_offline/strategy_arrival_slot
5,T31,scen_1500_92_sine_PO_L,105723,105723,0,offline_arrival_slot,44.358060,0.039000,2.498037,113759.837452,1,20,2.667034,0,0.000000,204449.805984,T31_sine_PO_L_offline_0_1116_1629_20/scen_1500_92_sine_PO_L/rust_offline/strategy_arrival_slot
6,T31,scen_1500_92_sine_PO_L,105723,103867,0,offline_bandit,53.993687,0.436000,3.910434,16539.297466,0,20,395.115289,0,0.000000,35950.060887,T31_sine_PO_L_offline_0_1116_1253_20/scen_1500_92_sine_PO_L/rust_offline/strategy_bandit
7,T31,scen_1500_92_sine_PO_L,105723,104013,0,offline_bandit,54.170870,0.371000,3.911290,91636.283458,0,20,330.605985,0,0.000000,199952.044948,T31_sine_PO_L_offline_0_1116_1629_20/scen_1500_92_sine_PO_L/rust_offline/strategy_bandit
8,T31,scen_1500_92_sine_PO_L,105723,102289,0,offline_hazard,78.393250,0.060000,4.003021,7543.356522,0,20,22.315869,0,0.000000,34912.036785,T31_sine_PO_L_offline_0_1116_1253_20/scen_1500_92_sine_PO_L/rust_offline/strategy_hazard
9,T31,scen_1500_92_sine_PO_L,105723,101423,0,offline_hazard,70.000511,0.055000,4.003579,58358.294877,0,20,20.409836,0,0.000000,194530.964216,T31_sine_PO_L_offline_0_1116_1629_20/scen_1500_92_sine_PO_L/rust_offline/strategy_hazard


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T32,scen_1500_92_sine_PS_shift0p25_L,105723,102659,1,dp,74.779747,12.525000,3.998432,55763.274621,296,20,0.018460,20,18.304263,221105.134925,T32_sine_PS_shift0p25_L_dp_0_1117_1010_20/scen_1500_92_sine_PS_shift0p25_L/rust_dp
1,T32,scen_1500_92_sine_PS_shift0p25_L,105723,104548,0,offline_ant_colony,68.421004,11.642000,3.942806,70609.611890,0,20,11587.619956,0,0.000000,223596.759501,T32_sine_PS_shift0p25_L_offline_0_1117_1014_20/scen_1500_92_sine_PS_shift0p25_L/rust_offline/strategy_ant_colony
2,T32,scen_1500_92_sine_PS_shift0p25_L,105723,105723,0,offline_arrival_slot,44.506756,0.050000,2.498037,124997.466238,1,20,3.084733,0,0.000000,225248.079970,T32_sine_PS_shift0p25_L_offline_0_1117_1014_20/scen_1500_92_sine_PS_shift0p25_L/rust_offline/strategy_arrival_slot
3,T32,scen_1500_92_sine_PS_shift0p25_L,105723,105341,0,offline_bandit,59.849201,0.412000,3.919272,90223.487101,0,20,366.191244,0,0.000000,224711.561487,T32_sine_PS_shift0p25_L_offline_0_1117_1014_20/scen_1500_92_sine_PS_shift0p25_L/rust_offline/strategy_bandit
4,T32,scen_1500_92_sine_PS_shift0p25_L,105723,101893,0,offline_hazard,74.428046,0.061000,4.003293,56166.131346,0,20,22.417321,0,0.000000,219639.572117,T32_sine_PS_shift0p25_L_offline_0_1117_1014_20/scen_1500_92_sine_PS_shift0p25_L/rust_offline/strategy_hazard
5,T32,scen_1500_92_sine_PS_shift0p25_L,105723,101130,0,offline_random_slot,52.587901,0.048000,2.505068,103564.680255,1,20,3.745531,0,0.000000,218435.130472,T32_sine_PS_shift0p25_L_offline_0_1117_1014_20/scen_1500_92_sine_PS_shift0p25_L/rust_offline/strategy_random_slot
6,T32,scen_1500_92_sine_PS_shift0p25_L,105723,102676,0,offline_singleton_best,74.731481,0.084000,3.997161,55887.635010,0,20,15.508407,0,0.000000,221174.956039,T32_sine_PS_shift0p25_L_offline_0_1117_1014_20/scen_1500_92_sine_PS_shift0p25_L/rust_offline/strategy_singleton_best
7,T32,scen_1500_92_sine_PS_shift0p25_L,105723,102676,0,offline_singleton_best_accurate,16.569184,0.054000,0.000000,184523.773409,0,20,12.666410,0,0.000000,221169.805446,T32_sine_PS_shift0p25_L_offline_0_1117_1014_20/scen_1500_92_sine_PS_shift0p25_L/rust_offline/strategy_singleton_best_accurate
8,T32,scen_1500_92_sine_PS_shift0p25_L,105723,105284,8,online_ant_colony,55.602297,4.573000,3.932055,99733.133196,0,20,2.017658,20,17.255439,224635.794977,T32_sine_PS_shift0p25_L_online_0_1117_1012_20/scen_1500_92_sine_PS_shift0p25_L/rust_online/online_ant_colony
9,T32,scen_1500_92_sine_PS_shift0p25_L,105723,105723,8,online_arrival_slot,44.415458,3.053000,2.503145,125203.112625,1,20,0.001106,20,12.288671,225248.079970,T32_sine_PS_shift0p25_L_perrequest_0_1117_1012_20/scen_1500_92_sine_PS_shift0p25_L/rust_online/online_arrival_slot


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T33,scen_1800_230_FLAT,129600,125111,1,dp,72.554354,12.675000,3.998949,69880.340130,302,20,0.014318,20,18.322293,254613.575072,T33_FLAT_dp_0_1117_1110_20/scen_1800_230_FLAT/rust_dp
1,T33,scen_1800_230_FLAT,129600,122763,0,offline_ant_colony,66.242809,12.663000,3.927568,84856.948280,0,20,12615.070780,0,0.000000,251374.436020,T33_FLAT_offline_0_1117_1113_20/scen_1800_230_FLAT/rust_offline/strategy_ant_colony
2,T33,scen_1800_230_FLAT,129600,129600,0,offline_arrival_slot,44.444949,0.049000,2.498283,144724.298243,0,20,3.450271,0,0.000000,260506.101492,T33_FLAT_offline_0_1117_1113_20/scen_1800_230_FLAT/rust_offline/strategy_arrival_slot
3,T33,scen_1800_230_FLAT,129600,126585,0,offline_bandit,59.383063,0.455000,3.946301,103899.618749,0,20,398.588468,0,0.000000,255803.675672,T33_FLAT_offline_0_1117_1113_20/scen_1800_230_FLAT/rust_offline/strategy_bandit
4,T33,scen_1800_230_FLAT,129600,124233,0,offline_hazard,70.889390,0.078000,3.996664,73682.572600,0,20,26.793924,0,0.000000,253112.433514,T33_FLAT_offline_0_1117_1113_20/scen_1800_230_FLAT/rust_offline/strategy_hazard
5,T33,scen_1800_230_FLAT,129600,123321,0,offline_random_slot,46.043738,0.046000,2.492986,135907.452191,0,20,4.061388,0,0.000000,251884.484395,T33_FLAT_offline_0_1117_1113_20/scen_1800_230_FLAT/rust_offline/strategy_random_slot
6,T33,scen_1800_230_FLAT,129600,125100,0,offline_singleton_best,72.724709,0.073000,3.997962,69444.240055,0,20,18.282691,0,0.000000,254604.942444,T33_FLAT_offline_0_1117_1113_20/scen_1800_230_FLAT/rust_offline/strategy_singleton_best
7,T33,scen_1800_230_FLAT,129600,125092,0,offline_singleton_best_accurate,9.509393,0.061000,0.000000,230381.073978,0,20,14.090353,0,0.000000,254591.147011,T33_FLAT_offline_0_1117_1113_20/scen_1800_230_FLAT/rust_offline/strategy_singleton_best_accurate
8,T33,scen_1800_230_FLAT,129600,129223,8,online_ant_colony,45.728677,6.135000,3.942487,141084.646906,0,20,2.054441,20,16.996683,259961.687734,T33_FLAT_online_0_1117_1111_20/scen_1800_230_FLAT/rust_online/online_ant_colony
9,T33,scen_1800_230_FLAT,129600,129600,8,online_arrival_slot,44.493771,3.747000,2.502758,144597.114179,6,20,0.001218,20,12.686486,260506.101492,T33_FLAT_perrequest_0_1117_1111_20/scen_1800_230_FLAT/rust_online/online_arrival_slot


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T20,scen_1800_230_sine_IP_L,128171,117092,1,dp,23.189349,12.681000,0.000000,220341.539242,0,20,0.006289,20,18.206157,286863.262942,T20_sine_IP_L_dp_0_1111_1620_20/scen_1800_230_sine_IP_L/rust_dp
1,T21,scen_1800_230_sine_IP_L,128171,117740,1,dp,22.956912,12.405000,0.000000,203944.322775,0,20,0.005972,20,18.072744,264714.627817,T21_sine_IP_L_dp_0_1111_1624_20/scen_1800_230_sine_IP_L/rust_dp
2,T22,scen_1800_230_sine_IP_L,128171,117326,1,dp,31.947099,12.525000,0.000000,222062.584273,0,20,0.006046,20,17.995189,326308.769016,T22_sine_IP_L_dp_0_1111_1622_20/scen_1800_230_sine_IP_L/rust_dp
3,T23,scen_1800_230_sine_IP_L,128171,117218,1,dp,24.794831,13.527000,0.000000,220887.962121,0,20,0.006945,20,18.182206,293713.802724,T23_sine_IP_L_dp_0_1112_1622_20/scen_1800_230_sine_IP_L/rust_dp
4,T00,scen_1800_230_sine_IP_L,128171,117095,1,dp,76.796413,14.206000,3.997950,66564.233057,234,20,0.014515,20,18.534046,286870.439045,T00_sine_IP_L_dp_0_1110_1913_20/scen_1800_230_sine_IP_L/rust_dp
5,T03,scen_1800_230_sine_IP_L,128171,117216,1,dp,77.240189,13.620000,3.998388,66847.763366,255,20,0.014292,20,18.599715,293709.658587,T03_sine_IP_L_dp_4_1115_1703_20/scen_1800_230_sine_IP_L/rust_dp
6,T30,scen_1800_230_sine_IP_L,128171,117217,1,dp,77.280702,13.277000,3.999143,66729.132588,219,20,0.013675,20,18.374290,293711.239770,T30_sine_IP_L_dp_0_1115_2246_20/scen_1800_230_sine_IP_L/rust_dp
7,T01,scen_1800_230_sine_IP_L,128171,117218,1,dp,77.410990,11.571000,3.999194,66347.188423,191,1,0.011251,1,1.000000,293714.453173,T01_sine_IP_L_dp_0_1114_1157_1/scen_1800_230_sine_IP_L/rust_dp
8,T02,scen_1800_230_sine_IP_L,128171,117218,1,dp,77.410990,11.623000,3.999194,66347.188423,191,1,0.011096,1,1.000000,293714.453173,T02_sine_IP_L_dp_0_1113_0004_1/scen_1800_230_sine_IP_L/rust_dp
9,T03,scen_1800_230_sine_IP_L,128171,117218,1,dp,77.410990,11.461000,3.999194,66347.188423,191,1,0.011275,1,1.000000,293714.453173,T03_sine_IP_L_dp_0_1115_1659_1/scen_1800_230_sine_IP_L/rust_dp


... (2 outputs left out)

,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T33,scen_2000_120_FLAT,144000,139456,1,dp,72.282763,15.021000,3.998896,79210.214784,226,20,0.014507,20,18.513002,285779.620674,T33_FLAT_dp_0_1117_1110_20/scen_2000_120_FLAT/rust_dp
1,T33,scen_2000_120_FLAT,144000,135158,0,offline_ant_colony,60.357826,14.922000,3.876981,110825.128790,0,20,14856.455776,0,0.000000,279563.698594,T33_FLAT_offline_0_1117_1113_20/scen_2000_120_FLAT/rust_offline/strategy_ant_colony
2,T33,scen_2000_120_FLAT,144000,144000,0,offline_arrival_slot,44.297642,0.058000,2.490729,162695.024017,0,20,3.742456,0,0.000000,292079.241609,T33_FLAT_offline_0_1117_1113_20/scen_2000_120_FLAT/rust_offline/strategy_arrival_slot
3,T33,scen_2000_120_FLAT,144000,142120,0,offline_bandit,56.283361,0.535000,3.883795,126493.055988,0,20,469.801538,0,0.000000,289347.620516,T33_FLAT_offline_0_1117_1113_20/scen_2000_120_FLAT/rust_offline/strategy_bandit
4,T33,scen_2000_120_FLAT,144000,137831,0,offline_hazard,70.617425,0.095000,3.997958,83241.774870,0,20,38.597974,0,0.000000,283303.197953,T33_FLAT_offline_0_1117_1113_20/scen_2000_120_FLAT/rust_offline/strategy_hazard
5,T33,scen_2000_120_FLAT,144000,136891,0,offline_random_slot,46.009511,0.063000,2.502045,152245.729084,0,20,4.288911,0,0.000000,281986.199965,T33_FLAT_offline_0_1117_1113_20/scen_2000_120_FLAT/rust_offline/strategy_random_slot
6,T33,scen_2000_120_FLAT,144000,139462,0,offline_singleton_best,72.459410,0.102000,3.999763,78715.252494,0,20,26.597370,0,0.000000,285815.416046,T33_FLAT_offline_0_1117_1113_20/scen_2000_120_FLAT/rust_offline/strategy_singleton_best
7,T33,scen_2000_120_FLAT,144000,139459,0,offline_singleton_best_accurate,8.232446,0.095000,0.000000,262282.029567,0,20,24.428610,0,0.000000,285811.290026,T33_FLAT_offline_0_1117_1113_20/scen_2000_120_FLAT/rust_offline/strategy_singleton_best_accurate
8,T33,scen_2000_120_FLAT,144000,142712,8,online_ant_colony,49.466198,5.682000,3.928822,146650.670515,0,20,1.684285,20,17.363186,290203.120516,T33_FLAT_online_0_1117_1111_20/scen_2000_120_FLAT/rust_online/online_ant_colony
9,T33,scen_2000_120_FLAT,144000,144000,8,online_arrival_slot,44.498333,4.195000,2.504566,162108.848845,1,20,0.001156,20,12.613428,292079.241609,T33_FLAT_perrequest_0_1117_1111_20/scen_2000_120_FLAT/rust_online/online_arrival_slot


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T20,scen_2000_120_sine_IP_L,146276,132741,1,dp,23.191030,16.213000,0.000000,256007.878161,0,20,0.006469,20,18.339453,333304.662590,T20_sine_IP_L_dp_0_1111_1620_20/scen_2000_120_sine_IP_L/rust_dp
1,T21,scen_2000_120_sine_IP_L,146276,132846,1,dp,22.660166,15.980000,0.000000,237641.737330,0,20,0.006367,20,18.360228,307269.520340,T21_sine_IP_L_dp_0_1111_1624_20/scen_2000_120_sine_IP_L/rust_dp
2,T22,scen_2000_120_sine_IP_L,146276,132542,1,dp,32.963821,16.091000,0.000000,259112.069609,0,20,0.006506,20,18.388054,386525.714653,T22_sine_IP_L_dp_0_1111_1622_20/scen_2000_120_sine_IP_L/rust_dp
3,T23,scen_2000_120_sine_IP_L,146276,132740,1,dp,27.162611,16.352000,0.000000,256659.553163,0,20,0.006811,20,18.493143,352373.359083,T23_sine_IP_L_dp_0_1112_1622_20/scen_2000_120_sine_IP_L/rust_dp
4,T03,scen_2000_120_sine_IP_L,146276,132742,1,dp,78.007653,16.846000,3.997002,77496.394074,164,20,0.014421,20,18.883387,352378.920084,T03_sine_IP_L_dp_4_1115_1703_20/scen_2000_120_sine_IP_L/rust_dp
5,T01,scen_2000_120_sine_IP_L,146276,132741,1,dp,77.987245,17.554000,3.998821,77567.697020,294,10,0.015067,10,9.786705,352376.139583,T01_sine_IP_L_dp_1_1114_1403_10/scen_2000_120_sine_IP_L/rust_dp
6,T01,scen_2000_120_sine_IP_L,146276,132740,1,dp,77.946421,16.757000,3.999491,77710.937723,391,20,0.014498,20,18.611639,352373.359083,T01_sine_IP_L_dp_2_1114_1428_20/scen_2000_120_sine_IP_L/rust_dp
7,T00,scen_2000_120_sine_IP_L,146276,132740,1,dp,76.828385,17.172000,3.999567,77231.429902,162,20,0.014132,20,18.625201,333301.882089,T00_sine_IP_L_dp_0_1110_1913_20/scen_2000_120_sine_IP_L/rust_dp
8,T03,scen_2000_120_sine_IP_L,146276,132740,1,dp,78.097803,16.744000,3.999604,77177.508649,85,5,0.013657,5,4.980483,352373.359083,T03_sine_IP_L_dp_1_1115_1700_5/scen_2000_120_sine_IP_L/rust_dp
9,T01,scen_2000_120_sine_IP_L,146276,132741,1,dp,78.141849,14.383000,3.999669,77022.737391,48,1,0.011033,1,1.000000,352375.359939,T01_sine_IP_L_dp_0_1114_1157_1/scen_2000_120_sine_IP_L/rust_dp


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T31,scen_2000_120_sine_PO_L,146276,140840,1,dp,68.996553,16.839000,3.997533,89391.823965,217,20,0.014551,20,18.710342,288328.657782,T31_sine_PO_L_dp_0_1116_1625_20/scen_2000_120_sine_PO_L/rust_dp
1,T31,scen_2000_120_sine_PO_L,146276,142850,1,dp,84.549291,16.675000,3.999842,8584.683775,218,20,0.014021,20,18.856872,55561.746118,T31_sine_PO_L_dp_0_1116_1249_20/scen_2000_120_sine_PO_L/rust_dp
2,T31,scen_2000_120_sine_PO_L,146276,143868,0,offline_ant_colony,71.831529,14.676000,3.922797,15890.711812,0,20,14613.625427,0,0.000000,56413.114690,T31_sine_PO_L_offline_0_1116_1253_20/scen_2000_120_sine_PO_L/rust_offline/strategy_ant_colony
3,T31,scen_2000_120_sine_PO_L,146276,142526,0,offline_ant_colony,58.278057,15.101000,3.931721,122606.366266,0,20,15037.656386,0,0.000000,293865.430267,T31_sine_PO_L_offline_0_1116_1629_20/scen_2000_120_sine_PO_L/rust_offline/strategy_ant_colony
4,T31,scen_2000_120_sine_PO_L,146276,146276,0,offline_arrival_slot,44.373610,0.057000,2.491540,32799.894102,0,20,3.584657,0,0.000000,58964.627850,T31_sine_PO_L_offline_0_1116_1253_20/scen_2000_120_sine_PO_L/rust_offline/strategy_arrival_slot
5,T31,scen_2000_120_sine_PO_L,146276,146276,0,offline_arrival_slot,44.324297,0.063000,2.491540,168153.426137,0,20,3.874348,0,0.000000,302022.993981,T31_sine_PO_L_offline_0_1116_1629_20/scen_2000_120_sine_PO_L/rust_offline/strategy_arrival_slot
6,T31,scen_2000_120_sine_PO_L,146276,143898,0,offline_bandit,47.746288,0.576000,3.868174,154502.659403,0,20,505.492583,0,0.000000,295677.866379,T31_sine_PO_L_offline_0_1116_1629_20/scen_2000_120_sine_PO_L/rust_offline/strategy_bandit
7,T31,scen_2000_120_sine_PO_L,146276,145351,0,offline_bandit,64.220799,0.551000,3.926409,20701.974738,0,20,493.427738,0,0.000000,57860.360163,T31_sine_PO_L_offline_0_1116_1253_20/scen_2000_120_sine_PO_L/rust_offline/strategy_bandit
8,T31,scen_2000_120_sine_PO_L,146276,140284,0,offline_hazard,69.971152,0.084000,3.998086,86431.019733,0,20,28.759886,0,0.000000,287826.620987,T31_sine_PO_L_offline_0_1116_1629_20/scen_2000_120_sine_PO_L/rust_offline/strategy_hazard
9,T31,scen_2000_120_sine_PO_L,146276,141414,0,offline_hazard,77.947140,0.083000,3.998402,12085.727015,0,20,29.243993,0,0.000000,54803.444718,T31_sine_PO_L_offline_0_1116_1253_20/scen_2000_120_sine_PO_L/rust_offline/strategy_hazard


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T32,scen_2000_120_sine_PS_shift0p25_L,146276,141577,1,dp,74.864230,18.706000,3.999449,82937.703367,191,20,0.016412,20,18.729722,329958.871898,T32_sine_PS_shift0p25_L_dp_0_1117_1010_20/scen_2000_120_sine_PS_shift0p25_L/rust_dp
1,T32,scen_2000_120_sine_PS_shift0p25_L,146276,146169,0,offline_ant_colony,73.642515,15.619000,4.000010,88653.850503,0,20,15552.929670,0,0.000000,336351.709441,T32_sine_PS_shift0p25_L_offline_0_1117_1014_20/scen_2000_120_sine_PS_shift0p25_L/rust_offline/strategy_ant_colony
2,T32,scen_2000_120_sine_PS_shift0p25_L,146276,146276,0,offline_arrival_slot,44.283343,0.063000,2.491540,187504.337619,0,20,4.015683,0,0.000000,336531.923429,T32_sine_PS_shift0p25_L_offline_0_1117_1014_20/scen_2000_120_sine_PS_shift0p25_L/rust_offline/strategy_arrival_slot
3,T32,scen_2000_120_sine_PS_shift0p25_L,146276,144897,0,offline_bandit,58.459734,0.575000,3.887072,138984.713240,0,20,516.016605,0,0.000000,334578.292835,T32_sine_PS_shift0p25_L_offline_0_1117_1014_20/scen_2000_120_sine_PS_shift0p25_L/rust_offline/strategy_bandit
4,T32,scen_2000_120_sine_PS_shift0p25_L,146276,140831,0,offline_hazard,74.495484,0.094000,3.997948,83674.756100,0,20,44.641418,0,0.000000,328078.195139,T32_sine_PS_shift0p25_L_offline_0_1117_1014_20/scen_2000_120_sine_PS_shift0p25_L/rust_offline/strategy_hazard
5,T32,scen_2000_120_sine_PS_shift0p25_L,146276,140042,0,offline_random_slot,52.793946,0.053000,2.501625,154263.167590,0,20,4.501532,0,0.000000,326786.827943,T32_sine_PS_shift0p25_L_offline_0_1117_1014_20/scen_2000_120_sine_PS_shift0p25_L/rust_offline/strategy_random_slot
6,T32,scen_2000_120_sine_PS_shift0p25_L,146276,141638,0,offline_singleton_best,74.852928,0.101000,3.997709,82895.292642,0,20,22.014143,0,0.000000,329641.931707,T32_sine_PS_shift0p25_L_offline_0_1117_1014_20/scen_2000_120_sine_PS_shift0p25_L/rust_offline/strategy_singleton_best
7,T32,scen_2000_120_sine_PS_shift0p25_L,146276,141585,0,offline_singleton_best_accurate,16.651175,0.072000,0.000000,274689.593718,0,20,16.480585,0,0.000000,329566.248020,T32_sine_PS_shift0p25_L_offline_0_1117_1014_20/scen_2000_120_sine_PS_shift0p25_L/rust_offline/strategy_singleton_best_accurate
8,T32,scen_2000_120_sine_PS_shift0p25_L,146276,145794,8,online_ant_colony,49.364956,5.536000,3.939565,170045.218850,0,20,1.653633,20,17.418377,335825.163725,T32_sine_PS_shift0p25_L_online_0_1117_1012_20/scen_2000_120_sine_PS_shift0p25_L/rust_online/online_ant_colony
9,T32,scen_2000_120_sine_PS_shift0p25_L,146276,146276,8,online_arrival_slot,44.545426,4.221000,2.504324,186622.343321,3,20,0.001065,20,13.556588,336531.923429,T32_sine_PS_shift0p25_L_perrequest_0_1117_1012_20/scen_2000_120_sine_PS_shift0p25_L/rust_online/online_arrival_slot


# Test 00 - Tuning DP

In [61]:
df = df_all[df_all["path"].str.contains("^T00")]
pd.set_option("display.max_rows", 15)
display_by_scenario(df.sort_values(by=SORTING_CRITERIA))


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T00,scen_1500_92_sine_IP_L,105723,95895,1,dp,77.157229,11.335000,4.000078,52128.529802,239,20,0.015666,20,18.333197,228205.810929,T00_sine_IP_L_dp_0_1110_1913_20/scen_1500_92_sine_IP_L/rust_dp
1,T00,scen_1500_92_sine_IP_L,105723,95920,4,dp,77.029512,47.480000,4.000547,52439.343861,1346,20,32.129292,20,18.450803,228290.073447,T00_sine_IP_L_dp_0_1110_1913_20/scen_1500_92_sine_IP_L/rust_dp
2,T00,scen_1500_92_sine_IP_L,105723,95939,6,dp,76.659059,240.304000,4.000302,53303.171693,4001,20,259.821556,20,19.186612,228367.704178,T00_sine_IP_L_dp_0_1110_1913_20/scen_1500_92_sine_IP_L/rust_dp
3,T00,scen_1500_92_sine_IP_L,105723,95961,8,dp,75.089204,566.509000,3.995712,56911.279716,13297,20,821.097378,20,19.245761,228460.302948,T00_sine_IP_L_dp_0_1110_1913_20/scen_1500_92_sine_IP_L/rust_dp
4,T00,scen_1500_92_sine_IP_L,105723,96008,22,dp,74.397038,1854.377000,3.967091,58516.908337,15626,20,7112.797170,20,18.587613,228555.232695,T00_sine_IP_L_dp_0_1110_1913_20/scen_1500_92_sine_IP_L/rust_dp


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T00,scen_1800_230_sine_IP_L,128171,117095,1,dp,76.796413,14.206000,3.997950,66564.233057,234,20,0.014515,20,18.534046,286870.439045,T00_sine_IP_L_dp_0_1110_1913_20/scen_1800_230_sine_IP_L/rust_dp
1,T00,scen_1800_230_sine_IP_L,128171,117091,4,dp,76.769153,57.457000,4.003830,66639.789623,1532,20,32.617094,20,18.703884,286859.065447,T00_sine_IP_L_dp_0_1110_1913_20/scen_1800_230_sine_IP_L/rust_dp
2,T00,scen_1800_230_sine_IP_L,128171,117091,6,dp,75.946651,325.580000,4.003531,69000.141175,6739,20,292.962515,20,19.332303,286862.928450,T00_sine_IP_L_dp_0_1110_1913_20/scen_1800_230_sine_IP_L/rust_dp
3,T00,scen_1800_230_sine_IP_L,128171,117093,8,dp,74.641743,704.540000,3.995179,72747.769948,17338,20,848.275325,20,19.366806,286880.014579,T00_sine_IP_L_dp_0_1110_1913_20/scen_1800_230_sine_IP_L/rust_dp
4,T00,scen_1800_230_sine_IP_L,128171,117134,22,dp,74.136168,2226.664000,3.973782,74230.397852,19622,20,7153.369055,20,18.842885,287004.637693,T00_sine_IP_L_dp_0_1110_1913_20/scen_1800_230_sine_IP_L/rust_dp


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T00,scen_2000_120_sine_IP_L,146276,132740,1,dp,76.828385,17.172000,3.999567,77231.429902,162,20,0.014132,20,18.625201,333301.882089,T00_sine_IP_L_dp_0_1110_1913_20/scen_2000_120_sine_IP_L/rust_dp
1,T00,scen_2000_120_sine_IP_L,146276,132730,4,dp,76.910850,65.157000,4.009210,76955.817775,1024,20,32.744210,20,18.857850,333298.622083,T00_sine_IP_L_dp_0_1110_1913_20/scen_2000_120_sine_IP_L/rust_dp
2,T00,scen_2000_120_sine_IP_L,146276,132727,6,dp,76.126105,368.568000,4.003066,79570.967367,6493,20,292.157521,20,19.427802,333296.959305,T00_sine_IP_L_dp_0_1110_1913_20/scen_2000_120_sine_IP_L/rust_dp
3,T00,scen_2000_120_sine_IP_L,146276,132721,8,dp,75.050686,772.036000,3.996316,83151.228435,15927,20,817.843251,20,19.445213,333280.614378,T00_sine_IP_L_dp_0_1110_1913_20/scen_2000_120_sine_IP_L/rust_dp
4,T00,scen_2000_120_sine_IP_L,146276,132711,22,dp,74.113024,2533.174000,3.971977,86282.889843,22263,20,7200.789948,20,19.024840,333306.173616,T00_sine_IP_L_dp_0_1110_1913_20/scen_2000_120_sine_IP_L/rust_dp


## Why saving does not grow along with `batch_size`

Increasing N (`batch_size`) allows to try more combinations of request placements, possibly finding more optimal schedules. However, higher N means more stale data: every worker receives an assignment snapshot of the current state; upon the commit of the solver proposed assignments, the error threshold may exceeded due to multiple sibling workers committing proposals made on outdated information. This yields following solver runs to fallback to the most accurate flavour, being incapable of finding alternative assignments which respect the error threshold, which have been already crossed.

20 workers each committing proposals for 22 requests means potentially up to 440 requests being proposed simultaneously, easily exceeding the threshold and causing fallbacks.\
Fallback is costful and completely outweighs the small potential gains from higher batch sizes.

Plus, larger N exponentially increases the combinations to consider, hence the processing time.

Furthermore, the flavours considered by the solver exhibit *forecasted* error and execution time, not real ones. In a real scenario, a request will be processed and the actual error may differ, as well as the execution time. Depending on the kind of task, they may vary significantly, easily outweighing the small benefits of larger batch sizes.
So, increasing batch size is not beneficial.


Technically, we could try to come up with different fallback strategies, but they did not seem to yield much improvement. Besides, we are talking nickels here.


## TEST 01 - Varying Batch size and parallelism

Here we vary both batch size and parallelism:
 - `batch_size = [1,2,3,4,5]`
 - `parallelism = [1,10,20]`

In [62]:
df = df_all[df_all["path"].str.contains("T01")]
df_par1 = df[df["paral"] == 1]
display_by_scenario(df_par1.head())

df_par10 = df[df["paral"] == 10]
display_by_scenario(df_par10.head())

df_par20 = df[df["paral"] == 20]
display_by_scenario(df_par20.head())


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T01,scen_1500_92_sine_IP_L,105723,95984,1,dp,78.172579,9.868000,3.998557,52015.133343,114,1,0.012727,1,1.000000,238301.782258,T01_sine_IP_L_dp_0_1114_1157_1/scen_1500_92_sine_IP_L/rust_dp
1,T01,scen_1500_92_sine_IP_L,105723,95984,2,dp,78.617261,14.215000,4.005980,50955.447819,29,1,0.170464,1,1.000000,238301.782258,T01_sine_IP_L_dp_0_1114_1157_1/scen_1500_92_sine_IP_L/rust_dp
2,T01,scen_1500_92_sine_IP_L,105723,95983,3,dp,78.707802,65.700000,4.006126,50738.847712,36,1,1.723119,1,1.000000,238297.835352,T01_sine_IP_L_dp_0_1114_1157_1/scen_1500_92_sine_IP_L/rust_dp
3,T01,scen_1500_92_sine_IP_L,105723,95983,4,dp,78.663018,322.397000,4.001229,50845.566452,12,1,11.998135,1,1.000000,238297.835352,T01_sine_IP_L_dp_0_1114_1157_1/scen_1500_92_sine_IP_L/rust_dp
4,T01,scen_1500_92_sine_IP_L,105723,95983,5,dp,78.869023,1742.340000,4.010788,50354.661904,11,1,82.030177,1,1.000000,238297.835352,T01_sine_IP_L_dp_0_1114_1157_1/scen_1500_92_sine_IP_L/rust_dp


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T01,scen_1500_92_sine_IP_L,105723,95983,1,dp,78.092703,10.354000,3.998937,52204.290952,97,10,0.013486,10,9.689440,238296.358989,T01_sine_IP_L_dp_1_1114_1403_10/scen_1500_92_sine_IP_L/rust_dp
1,T01,scen_1500_92_sine_IP_L,105723,95984,2,dp,78.069628,7.301000,3.999495,52259.869115,256,10,0.198174,10,9.639796,238299.057327,T01_sine_IP_L_dp_1_1114_1403_10/scen_1500_92_sine_IP_L/rust_dp
2,T01,scen_1500_92_sine_IP_L,105723,95993,3,dp,78.145496,10.421000,4.005839,52085.707597,332,10,2.160445,10,9.681807,238329.392908,T01_sine_IP_L_dp_1_1114_1403_10/scen_1500_92_sine_IP_L/rust_dp
3,T01,scen_1500_92_sine_IP_L,105723,95998,4,dp,78.174026,56.478000,4.000839,52022.060566,417,10,19.997617,10,9.606948,238349.319738,T01_sine_IP_L_dp_1_1114_1403_10/scen_1500_92_sine_IP_L/rust_dp
4,T01,scen_1500_92_sine_IP_L,105723,95998,5,dp,78.463541,298.357000,4.001516,51332.956927,372,10,132.362294,10,9.442437,238353.747268,T01_sine_IP_L_dp_1_1114_1403_10/scen_1500_92_sine_IP_L/rust_dp


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T01,scen_1500_92_sine_IP_L,105723,95983,1,dp,78.055852,10.345000,4.000813,52292.644839,175,20,0.013635,20,18.270554,238298.819074,T01_sine_IP_L_dp_2_1114_1428_20/scen_1500_92_sine_IP_L/rust_dp
1,T01,scen_1500_92_sine_IP_L,105723,95983,2,dp,78.096171,7.234000,4.001907,52196.184767,214,20,0.194850,20,17.307251,238297.077866,T01_sine_IP_L_dp_2_1114_1428_20/scen_1500_92_sine_IP_L/rust_dp
2,T01,scen_1500_92_sine_IP_L,105723,96006,3,dp,78.019346,9.417000,4.002302,52398.653601,929,20,2.950019,20,18.379166,238385.322516,T01_sine_IP_L_dp_2_1114_1428_20/scen_1500_92_sine_IP_L/rust_dp
3,T01,scen_1500_92_sine_IP_L,105723,96006,4,dp,78.028825,47.927000,3.999099,52372.249447,1159,20,32.481874,20,18.505508,238367.993881,T01_sine_IP_L_dp_2_1114_1428_20/scen_1500_92_sine_IP_L/rust_dp
4,T01,scen_1500_92_sine_IP_L,105723,96017,5,dp,78.290090,271.362000,3.999318,51756.020329,1141,20,229.226864,20,17.990165,238398.131487,T01_sine_IP_L_dp_2_1114_1428_20/scen_1500_92_sine_IP_L/rust_dp


In [63]:
comparison_parallelisms = [1, 10, 20]
parallelism_comparison = compare_metrics_wide(
    df_all,
    row_keys=["scenario_id", "batch"],
    comparison_column="paral",
    comparison_values=comparison_parallelisms,
    metric_sources=comparison_metric_sources,
    filters={"run_id": "T01", "policy": "dp"},
    labels={1: "P1", 10: "P10", 20: "P20"},
    key_renames={"batch": "batch_size"},
)
display_by_scenario(parallelism_comparison)

,scenario_id,batch_size,elapsed_s_P1,elapsed_s_P10,elapsed_s_P20,carb_saving_P1,carb_saving_P10,carb_saving_P20,error_P1,error_P10,error_P20,solver_ms_P1,solver_ms_P10,solver_ms_P20,carb_cost_P1,carb_cost_P10,carb_cost_P20
0,scen_1500_92_sine_IP_L,1,9.868000,10.354000,10.345000,78.172579,78.092703,78.055852,3.998557,3.998937,4.000813,0.012727,0.013486,0.013635,52015.133343,52204.290952,52292.644839
1,scen_1500_92_sine_IP_L,2,14.215000,7.301000,7.234000,78.617261,78.069628,78.096171,4.005980,3.999495,4.001907,0.170464,0.198174,0.194850,50955.447819,52259.869115,52196.184767
2,scen_1500_92_sine_IP_L,3,65.700000,10.421000,9.417000,78.707802,78.145496,78.019346,4.006126,4.005839,4.002302,1.723119,2.160445,2.950019,50738.847712,52085.707597,52398.653601
3,scen_1500_92_sine_IP_L,4,322.397000,56.478000,47.927000,78.663018,78.174026,78.028825,4.001229,4.000839,3.999099,11.998135,19.997617,32.481874,50845.566452,52022.060566,52372.249447
4,scen_1500_92_sine_IP_L,5,1742.340000,298.357000,271.362000,78.869023,78.463541,78.290090,4.010788,4.001516,3.999318,82.030177,132.362294,229.226864,50354.661904,51332.956927,51756.020329


,scenario_id,batch_size,elapsed_s_P1,elapsed_s_P10,elapsed_s_P20,carb_saving_P1,carb_saving_P10,carb_saving_P20,error_P1,error_P10,error_P20,solver_ms_P1,solver_ms_P10,solver_ms_P20,carb_cost_P1,carb_cost_P10,carb_cost_P20
0,scen_1800_230_sine_IP_L,1,11.571000,13.670000,13.767000,77.410990,77.336074,77.316039,3.999194,3.999966,4.000793,0.011251,0.013879,0.014132,66347.188423,66566.255170,66626.262495
1,scen_1800_230_sine_IP_L,2,16.920000,9.086000,9.114000,77.939487,77.386920,77.443621,4.002180,4.001147,4.002807,0.164731,0.196438,0.190661,64794.916383,66417.496581,66251.998084
2,scen_1800_230_sine_IP_L,3,71.964000,12.649000,11.550000,78.145770,77.327649,77.300871,4.008855,4.004658,4.008842,1.555626,2.197293,2.897423,64189.031055,66591.796340,66672.635272
3,scen_1800_230_sine_IP_L,4,366.895000,68.957000,57.806000,78.346124,77.450325,77.289149,4.025384,4.001066,4.002107,11.256265,20.284926,32.748060,63600.563317,66231.280888,66702.800257
4,scen_1800_230_sine_IP_L,5,1942.713000,363.460000,325.372000,78.209546,77.783503,77.530818,4.009602,4.003451,4.001753,75.459816,134.660767,231.215996,64001.712824,65253.801659,65990.435993


,scenario_id,batch_size,elapsed_s_P1,elapsed_s_P10,elapsed_s_P20,carb_saving_P1,carb_saving_P10,carb_saving_P20,error_P1,error_P10,error_P20,solver_ms_P1,solver_ms_P10,solver_ms_P20,carb_cost_P1,carb_cost_P10,carb_cost_P20
0,scen_2000_120_sine_IP_L,1,14.383000,17.554000,16.757000,78.141849,77.987245,77.946421,3.999669,3.998821,3.999491,0.011033,0.015067,0.014498,77022.737391,77567.697020,77710.937723
1,scen_2000_120_sine_IP_L,2,19.400000,11.111000,10.759000,78.568089,78.092223,78.085420,4.000780,4.000520,4.003503,0.161983,0.196874,0.190010,75520.773235,77196.729966,77221.310853
2,scen_2000_120_sine_IP_L,3,78.391000,14.895000,13.333000,78.779914,78.075833,77.854402,4.006185,4.005402,4.001055,1.478727,2.227423,2.863034,74774.355744,77253.778853,78036.492635
3,scen_2000_120_sine_IP_L,4,426.926000,80.628000,65.658000,78.903808,78.247259,78.038490,4.008795,4.007730,4.003711,11.476806,20.974723,32.976601,74337.359723,76650.405241,77385.316402
4,scen_2000_120_sine_IP_L,5,2176.630000,427.525000,353.537000,78.833230,78.644713,78.327386,3.999198,4.005782,4.000418,74.077949,140.016149,223.376865,74586.482679,75250.390997,76363.364775


### Parallelism effect
Parallelism leads the solver time increase, since there is more contention, but in general provides a considerable speedup on the total elapsed time (6.4x, 540% faster). Note that acquiring locks is not considered in this measure, it's just the pure solver time.\
Parallelism implies a negligible performance degration in saving (<1%).

There is not much difference between parallelism=10 and 20, perhaps due to the real physical amount of threads actually available to WSL.


Parallelism though seems uneffective for N=1. That's probably because the most work is serial anyway. 0.013ms per solve is very small, and for about 100k solves it adds up to 1.3s, out of the total 10s elapsed.
That's because the assignment committing is lock guarded, hence serial, and parallelism can't help much. The same goes for writing out results as the run progresses. In fact, parallelism and lock management end up introducing more overheap than benefit.


## T01/T03 - More testing on N=1 
T03 is another test for N=1, to ensure parallelism really does not speed up the execution and that it was not a random case what we saw before.

In [64]:
df = df_all[df_all["path"].str.contains("T03")]
display_by_scenario(df.sort_values(["scenario_id","paral"]))

,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T03,scen_1500_92_sine_IP_L,105723,95984,1,dp,78.172579,8.976000,3.998557,52015.133343,114,1,0.011279,1,1.000000,238301.782258,T03_sine_IP_L_dp_0_1115_1659_1/scen_1500_92_sine_IP_L/rust_dp
1,T03,scen_1500_92_sine_IP_L,105723,95983,1,dp,78.015795,10.584000,3.996619,52387.884439,204,5,0.014391,5,4.976283,238297.835352,T03_sine_IP_L_dp_1_1115_1700_5/scen_1500_92_sine_IP_L/rust_dp
2,T03,scen_1500_92_sine_IP_L,105723,95985,1,dp,77.987201,10.547000,3.998125,52457.652617,257,10,0.014496,10,9.768295,238305.236524,T03_sine_IP_L_dp_2_1115_1701_10/scen_1500_92_sine_IP_L/rust_dp
3,T03,scen_1500_92_sine_IP_L,105723,95984,1,dp,78.051609,10.716000,4.000797,52303.299473,235,14,0.014368,14,13.413859,238301.289618,T03_sine_IP_L_dp_3_1115_1702_14/scen_1500_92_sine_IP_L/rust_dp
4,T03,scen_1500_92_sine_IP_L,105723,95983,1,dp,78.023074,10.519000,3.999458,52370.430352,267,20,0.014169,20,18.308222,238297.342712,T03_sine_IP_L_dp_4_1115_1703_20/scen_1500_92_sine_IP_L/rust_dp


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T03,scen_1800_230_sine_IP_L,128171,117218,1,dp,77.410990,11.461000,3.999194,66347.188423,191,1,0.011275,1,1.000000,293714.453173,T03_sine_IP_L_dp_0_1115_1659_1/scen_1800_230_sine_IP_L/rust_dp
1,T03,scen_1800_230_sine_IP_L,128171,117218,1,dp,77.344020,13.543000,4.000239,66544.169692,136,5,0.013850,5,4.980782,293715.691325,T03_sine_IP_L_dp_1_1115_1700_5/scen_1800_230_sine_IP_L/rust_dp
2,T03,scen_1800_230_sine_IP_L,128171,117216,1,dp,77.304189,13.533000,3.999561,66659.565445,212,10,0.013887,10,9.795152,293708.676815,T03_sine_IP_L_dp_2_1115_1701_10/scen_1800_230_sine_IP_L/rust_dp
3,T03,scen_1800_230_sine_IP_L,128171,117217,1,dp,77.356439,14.494000,4.000700,66506.624551,148,14,0.015206,14,13.492047,293710.983389,T03_sine_IP_L_dp_3_1115_1702_14/scen_1800_230_sine_IP_L/rust_dp
4,T03,scen_1800_230_sine_IP_L,128171,117216,1,dp,77.240189,13.620000,3.998388,66847.763366,255,20,0.014292,20,18.599715,293709.658587,T03_sine_IP_L_dp_4_1115_1703_20/scen_1800_230_sine_IP_L/rust_dp


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T03,scen_2000_120_sine_IP_L,146276,132741,1,dp,78.141849,14.190000,3.999669,77022.737391,48,1,0.010769,1,1.000000,352375.359939,T03_sine_IP_L_dp_0_1115_1659_1/scen_2000_120_sine_IP_L/rust_dp
1,T03,scen_2000_120_sine_IP_L,146276,132740,1,dp,78.097803,16.744000,3.999604,77177.508649,85,5,0.013657,5,4.980483,352373.359083,T03_sine_IP_L_dp_1_1115_1700_5/scen_2000_120_sine_IP_L/rust_dp
2,T03,scen_2000_120_sine_IP_L,146276,132740,1,dp,78.074032,16.501000,4.001224,77261.270382,119,10,0.013565,10,9.797665,352373.359083,T03_sine_IP_L_dp_2_1115_1701_10/scen_2000_120_sine_IP_L/rust_dp
3,T03,scen_2000_120_sine_IP_L,146276,132740,1,dp,78.087769,17.497000,4.000603,77212.862823,93,14,0.014760,14,13.594387,352373.359083,T03_sine_IP_L_dp_3_1115_1702_14/scen_2000_120_sine_IP_L/rust_dp
4,T03,scen_2000_120_sine_IP_L,146276,132742,1,dp,78.007653,16.846000,3.997002,77496.394074,164,20,0.014421,20,18.883387,352378.920084,T03_sine_IP_L_dp_4_1115_1703_20/scen_2000_120_sine_IP_L/rust_dp


## TEST 02 - Batch size with no parallelism
How much can we gain from higher batch sizes if parallelism (hence fallbacks) cannot hinder performance?
*Spoiler*: More or less none.

In [65]:
df = df_all[df_all["path"].str.contains("T02")]
display_by_scenario(df)

,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T02,scen_1500_92_sine_IP_L,105723,95984,1,dp,78.172579,8.955000,3.998557,52015.133343,114,1,0.011184,1,1.000000,238301.782258,T02_sine_IP_L_dp_0_1113_0004_1/scen_1500_92_sine_IP_L/rust_dp
1,T02,scen_1500_92_sine_IP_L,105723,95983,4,dp,78.664686,296.469000,4.001229,50841.591559,12,1,11.030392,1,1.000000,238297.835352,T02_sine_IP_L_dp_0_1113_0004_1/scen_1500_92_sine_IP_L/rust_dp
2,T02,scen_1500_92_sine_IP_L,105723,95983,8,dp,78.667860,2756.135000,4.000214,50834.027938,16,1,207.856832,1,1.000000,238297.835352,T02_sine_IP_L_dp_0_1113_0004_1/scen_1500_92_sine_IP_L/rust_dp
3,T02,scen_1500_92_sine_IP_L,105723,95981,16,dp,78.583897,5892.239000,4.010325,51032.418300,0,1,886.823263,1,1.000000,238289.941540,T02_sine_IP_L_dp_0_1113_0004_1/scen_1500_92_sine_IP_L/rust_dp


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T02,scen_1800_230_sine_IP_L,128171,117218,1,dp,77.410990,11.623000,3.999194,66347.188423,191,1,0.011096,1,1.000000,293714.453173,T02_sine_IP_L_dp_0_1113_0004_1/scen_1800_230_sine_IP_L/rust_dp
1,T02,scen_1800_230_sine_IP_L,128171,117218,4,dp,78.345841,342.947000,4.025384,63601.394562,22,1,10.519401,1,1.000000,293714.453173,T02_sine_IP_L_dp_0_1113_0004_1/scen_1800_230_sine_IP_L/rust_dp
2,T02,scen_1800_230_sine_IP_L,128171,117217,8,dp,78.032267,3289.756000,3.999569,64522.116198,16,1,204.752282,1,1.000000,293713.128371,T02_sine_IP_L_dp_0_1113_0004_1/scen_1800_230_sine_IP_L/rust_dp
3,T02,scen_1800_230_sine_IP_L,128171,117215,16,dp,77.878002,7148.422000,4.006185,64973.963982,16,1,888.275584,1,1.000000,293707.489701,T02_sine_IP_L_dp_0_1113_0004_1/scen_1800_230_sine_IP_L/rust_dp


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T02,scen_2000_120_sine_IP_L,146276,132741,1,dp,78.141849,13.571000,3.999669,77022.737391,48,1,0.010210,1,1.000000,352375.359939,T02_sine_IP_L_dp_0_1113_0004_1/scen_2000_120_sine_IP_L/rust_dp
1,T02,scen_2000_120_sine_IP_L,146276,132740,4,dp,78.903960,403.166000,4.008795,74336.824081,32,1,10.835811,1,1.000000,352373.359083,T02_sine_IP_L_dp_0_1113_0004_1/scen_2000_120_sine_IP_L/rust_dp
2,T02,scen_2000_120_sine_IP_L,146276,132740,8,dp,78.763936,3763.469000,3.999830,74830.233120,16,1,205.239671,1,1.000000,352373.359083,T02_sine_IP_L_dp_0_1113_0004_1/scen_2000_120_sine_IP_L/rust_dp
3,T02,scen_2000_120_sine_IP_L,146276,132740,16,dp,78.458186,7991.733000,4.007138,75907.614250,0,1,870.657570,1,1.000000,352373.359083,T02_sine_IP_L_dp_0_1113_0004_1/scen_2000_120_sine_IP_L/rust_dp


Having no parallelism takes down the problem of fallbacks due to simultaneous commits. Fallbacks may occur only upon the sliding of the window, which is less frequent and is smoothed by the decayed past history.


However we can see that the benefit of N>1 with respect to N=1 is almost negligible, and surely would be even more negligible in practice, where the forecasted execution times (hence carbon cost) will be replaced by the actual ones, which (most likely) exhibit high variance.

Interestingly we can see that N=16 yields slightly lower carbon savings than N=8 or N=4. Seems counterintuitive, but actually it could be due to pruning: with higher N, the options grow exponentially, and there is a higher chance some potentially good options get pruned.

Furthermore,for N>1 there is huge overhead: elapsed time grows exponentially with N. We go from almost instantaneous solving (0.01ms) with N=1 to 880ms with N=16 for a single batch.
So, for high N, we would also have to consider the carbon cost of the overhead, which may outweigh the benefits, depending on the carbon cost of the tasks at hand.

## Conclusion

N=1 is simplest and yields the best results, and is extra fast.

# TEST 20|21|22|23 - Analyzing cap tiers

In [66]:
df = df_all[df_all["path"].str.contains("^T2")]
# Remove "_sine_IP_L" from scenario ids
df["scenario_id"] = df["scenario_id"].str.replace("_sine_IP_L", "")
pd.set_option("display.max_rows", 15)
display_by_scenario(df.sort_values(by=SORTING_CRITERIA))

/tmp/ipykernel_139186/781621322.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["scenario_id"] = df["scenario_id"].str.replace("_sine_IP_L", "")


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T20,scen_1500_92,105723,95899,1,dp,24.378407,9.929000,0.000000,172580.143867,0,20,0.006329,20,17.952006,228215.429815,T20_sine_IP_L_dp_0_1111_1620_20/scen_1500_92_sine_IP_L/rust_dp
1,T21,scen_1500_92,105723,96728,1,dp,24.056880,10.000000,0.000000,161051.007505,0,20,0.006371,20,17.879127,212067.936187,T21_sine_IP_L_dp_0_1111_1624_20/scen_1500_92_sine_IP_L/rust_dp
2,T22,scen_1500_92,105723,96052,1,dp,34.246391,9.831000,0.000000,174743.037433,0,20,0.006180,20,17.711413,265754.290321,T22_sine_IP_L_dp_0_1111_1622_20/scen_1500_92_sine_IP_L/rust_dp
3,T23,scen_1500_92,105723,95983,1,dp,27.318487,10.467000,0.000000,173196.326556,0,20,0.007110,20,18.083071,238294.882627,T23_sine_IP_L_dp_0_1112_1622_20/scen_1500_92_sine_IP_L/rust_dp


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T20,scen_1800_230,128171,117092,1,dp,23.189349,12.681000,0.000000,220341.539242,0,20,0.006289,20,18.206157,286863.262942,T20_sine_IP_L_dp_0_1111_1620_20/scen_1800_230_sine_IP_L/rust_dp
1,T21,scen_1800_230,128171,117740,1,dp,22.956912,12.405000,0.000000,203944.322775,0,20,0.005972,20,18.072744,264714.627817,T21_sine_IP_L_dp_0_1111_1624_20/scen_1800_230_sine_IP_L/rust_dp
2,T22,scen_1800_230,128171,117326,1,dp,31.947099,12.525000,0.000000,222062.584273,0,20,0.006046,20,17.995189,326308.769016,T22_sine_IP_L_dp_0_1111_1622_20/scen_1800_230_sine_IP_L/rust_dp
3,T23,scen_1800_230,128171,117218,1,dp,24.794831,13.527000,0.000000,220887.962121,0,20,0.006945,20,18.182206,293713.802724,T23_sine_IP_L_dp_0_1112_1622_20/scen_1800_230_sine_IP_L/rust_dp


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T20,scen_2000_120,146276,132741,1,dp,23.191030,16.213000,0.000000,256007.878161,0,20,0.006469,20,18.339453,333304.662590,T20_sine_IP_L_dp_0_1111_1620_20/scen_2000_120_sine_IP_L/rust_dp
1,T21,scen_2000_120,146276,132846,1,dp,22.660166,15.980000,0.000000,237641.737330,0,20,0.006367,20,18.360228,307269.520340,T21_sine_IP_L_dp_0_1111_1624_20/scen_2000_120_sine_IP_L/rust_dp
2,T22,scen_2000_120,146276,132542,1,dp,32.963821,16.091000,0.000000,259112.069609,0,20,0.006506,20,18.388054,386525.714653,T22_sine_IP_L_dp_0_1111_1622_20/scen_2000_120_sine_IP_L/rust_dp
3,T23,scen_2000_120,146276,132740,1,dp,27.162611,16.352000,0.000000,256659.553163,0,20,0.006811,20,18.493143,352373.359083,T23_sine_IP_L_dp_0_1112_1622_20/scen_2000_120_sine_IP_L/rust_dp


| TEST | SCENARIO | CAP TIERS (threshold, CI multiplier) |
|---|---|---|
| 20   | scen_1500_92 | (1500, 1.5x) / (3000, 5.0x) |
| 20   | scen_1800_230 | (1800, 1.5x) / (3600, 5.0x) |
| 20   | scen_2000_120 | (2000, 1.5x) / (4000, 5.0x) |
| 21   | scen_1500_92 | (2250, 1.5x) / (3000, 5.0x) |
| 21   | scen_1800_230 | (2700, 1.5x) / (3600, 5.0x) |
| 21   | scen_2000_120 | (3000, 1.5x) / (4000, 5.0x) |
| 22   | scen_1500_92 | (1500, 1.5x) / (2250, 5.0x) |
| 22   | scen_1800_230 | (1800, 1.5x) / (2700, 5.0x) |
| 22   | scen_2000_120 | (2000, 1.5x) / (3000, 5.0x) |
| 23   | scen_1500_92 | (1500, 1.5x) / (2625, 5.0x) |
| 23   | scen_1800_230 | (1800, 1.5x) / (3150, 5.0x) |
| 23   | scen_2000_120 | (2000, 1.5x) / (3500, 5.0x) |

So essentially, cap tiers are set based on the scenario's average load. For T20 tests, the first cap matches the average load, while the second its double. For T21, the first cap is 1.5x the average and the second 2x. For T22, the first matches the average and the second 1.5x. For T23, the first matches the average and the second is 1.75x the average.
Carbon intensity multiplier are always 1.5x and 5.0x.

Highest carbon saving (~33% vs 24%) is obtained for T22.
However, this happens mainly because the baseline cost is higher for T22.
By looking at the carbon cost alone T22 is always the highest.
T23 seems to exhibit the higher saving, with an absolute carbon cost lower than T22 and comparable to T20.
T21 exhibits the lowest absolute carbon cost, given that its cap tiers are both high.

Cap tiers are not really a configurable parameter in reality, they are bound to hardware resources.

These tests allow to state that the system should be able to handle under standard conditions loads up to 1.75x the average input load, before triggering a strong rebound effect.


# TEST 10 - Tuning Online and Swarm strategies

In [67]:
df = df_all[df_all["path"].str.contains("^T1")]
# Remove "_sine_IP_L" from scenario ids
df["scenario_id"] = df["scenario_id"].str.replace("_sine_IP_L", "")
pd.set_option("display.max_rows", 5)
display_by_scenario(df.sort_values(by=SORTING_CRITERIA))

/tmp/ipykernel_139186/856984455.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["scenario_id"] = df["scenario_id"].str.replace("_sine_IP_L", "")


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T11,scen_1500_92,105723,101453,1,online_ant_colony,50.014356,10.789000,3.855406,126767.856345,0,20,0.203392,20,18.842796,253608.530464,T11_sine_IP_L_online_0_1113_1222_20/scen_1500_92_sine_IP_L/rust_online/online_ant_colony
1,T13,scen_1500_92,105723,101760,1,online_ant_colony,49.600202,10.731000,3.856697,128263.518752,0,20,0.129207,20,18.554581,254492.126442,T13_sine_IP_L_online_0_1113_1740_20/scen_1500_92_sine_IP_L/rust_online/online_ant_colony
2,T12,scen_1500_92,105723,105263,1,online_ant_colony,45.990891,10.883000,3.948111,140970.605840,0,20,0.198371,20,19.005225,261012.650607,T12_sine_IP_L_online_0_1113_1705_20/scen_1500_92_sine_IP_L/rust_online/online_ant_colony
3,T12,scen_1500_92,105723,98311,8,online_ant_colony,52.461074,7.738000,3.811527,116775.046236,0,20,8.737613,20,18.934693,245640.900771,T12_sine_IP_L_online_0_1113_1705_20/scen_1500_92_sine_IP_L/rust_online/online_ant_colony
4,T11,scen_1500_92,105723,99710,8,online_ant_colony,51.733883,13.478000,3.814236,120577.549121,0,20,17.697190,20,18.989701,249818.210393,T11_sine_IP_L_online_0_1113_1222_20/scen_1500_92_sine_IP_L/rust_online/online_ant_colony
5,T14,scen_1500_92,105723,102015,8,online_ant_colony,50.775329,4.835000,3.841224,125465.428783,0,20,2.078546,20,17.443237,254883.223612,T14_sine_IP_L_online_2_1115_1748_20/scen_1500_92_sine_IP_L/rust_online/online_ant_colony
6,T14,scen_1500_92,105723,102388,8,online_ant_colony,50.921240,5.000000,3.879190,123869.934946,0,10,1.801890,10,9.571780,252390.106992,T14_sine_IP_L_online_1_1115_1748_10/scen_1500_92_sine_IP_L/rust_online/online_ant_colony
7,T10,scen_1500_92,105723,104045,8,online_ant_colony,49.262817,4.622000,3.891874,130133.339424,0,20,1.994317,20,17.118457,256485.147770,T10_sine_IP_L_online_0_1113_2330_20/scen_1500_92_sine_IP_L/rust_online/online_ant_colony
8,T13,scen_1500_92,105723,103934,8,online_ant_colony,50.729540,4.671000,3.894034,126354.587215,0,20,1.988514,20,17.582418,256451.000879,T13_sine_IP_L_online_0_1113_1740_20/scen_1500_92_sine_IP_L/rust_online/online_ant_colony
9,T14,scen_1500_92,105723,105141,8,online_ant_colony,52.516907,13.656000,3.989975,123864.794353,0,1,0.838445,1,1.000000,260860.835527,T14_sine_IP_L_online_0_1115_1746_1/scen_1500_92_sine_IP_L/rust_online/online_ant_colony


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T13,scen_1800_230,128171,119536,1,online_ant_colony,56.312952,13.770000,3.850430,131952.966157,0,20,0.126541,20,18.744623,302041.385981,T13_sine_IP_L_online_0_1113_1740_20/scen_1800_230_sine_IP_L/rust_online/online_ant_colony
1,T12,scen_1800_230,128171,124691,1,online_ant_colony,53.905672,14.082000,3.904853,142784.785082,0,20,0.189164,20,18.847296,309766.495670,T12_sine_IP_L_online_0_1113_1705_20/scen_1800_230_sine_IP_L/rust_online/online_ant_colony
2,T11,scen_1800_230,128171,127760,1,online_ant_colony,50.005038,13.868000,3.933058,159952.859676,0,20,0.202402,20,18.986639,319937.958360,T11_sine_IP_L_online_0_1113_1222_20/scen_1800_230_sine_IP_L/rust_online/online_ant_colony
3,T12,scen_1800_230,128171,121179,8,online_ant_colony,44.635777,8.134000,3.851988,169963.025151,0,20,7.583565,20,19.021170,306990.714551,T12_sine_IP_L_online_0_1113_1705_20/scen_1800_230_sine_IP_L/rust_online/online_ant_colony
4,T14,scen_1800_230,128171,121160,8,online_ant_colony,54.913248,5.660000,3.853355,136698.920400,0,10,1.665819,10,9.617680,303190.883602,T14_sine_IP_L_online_1_1115_1748_10/scen_1800_230_sine_IP_L/rust_online/online_ant_colony
5,T14,scen_1800_230,128171,123643,8,online_ant_colony,51.473263,4.786000,3.863037,151981.922185,0,20,1.580342,20,17.800381,313192.131929,T14_sine_IP_L_online_2_1115_1748_20/scen_1800_230_sine_IP_L/rust_online/online_ant_colony
6,T11,scen_1800_230,128171,124652,8,online_ant_colony,52.419843,16.660000,3.891955,147317.634423,0,20,18.364440,20,19.185264,309619.897400,T11_sine_IP_L_online_0_1113_1222_20/scen_1800_230_sine_IP_L/rust_online/online_ant_colony
7,T13,scen_1800_230,128171,126245,8,online_ant_colony,51.857080,5.541000,3.902749,151368.959848,0,20,1.929687,20,17.198139,314415.829788,T13_sine_IP_L_online_0_1113_1740_20/scen_1800_230_sine_IP_L/rust_online/online_ant_colony
8,T10,scen_1800_230,128171,126335,8,online_ant_colony,53.253764,5.232000,3.908715,147127.201299,0,20,1.763743,20,17.346429,314735.929042,T10_sine_IP_L_online_0_1113_2330_20/scen_1800_230_sine_IP_L/rust_online/online_ant_colony
9,T14,scen_1800_230,128171,127140,8,online_ant_colony,55.613596,15.394000,3.968086,141260.393367,0,1,0.777045,1,1.000000,318251.492571,T14_sine_IP_L_online_0_1115_1746_1/scen_1800_230_sine_IP_L/rust_online/online_ant_colony


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T13,scen_2000_120,146276,144799,1,online_ant_colony,44.449841,17.392000,3.930397,214091.301995,0,20,0.132268,20,18.941586,385401.781468,T13_sine_IP_L_online_0_1113_1740_20/scen_2000_120_sine_IP_L/rust_online/online_ant_colony
1,T11,scen_2000_120,146276,145304,1,online_ant_colony,52.190349,16.824000,3.938931,184622.136624,0,20,0.201145,20,19.134697,386160.812429,T11_sine_IP_L_online_0_1113_1222_20/scen_2000_120_sine_IP_L/rust_online/online_ant_colony
2,T12,scen_2000_120,146276,145755,1,online_ant_colony,49.419405,16.995000,3.949453,195645.537496,0,20,0.185441,20,19.029598,386799.593388,T12_sine_IP_L_online_0_1113_1705_20/scen_2000_120_sine_IP_L/rust_online/online_ant_colony
3,T11,scen_2000_120,146276,137064,8,online_ant_colony,54.574711,16.981000,3.818417,167100.777838,0,20,16.470374,20,19.223760,367858.483292,T11_sine_IP_L_online_0_1113_1222_20/scen_2000_120_sine_IP_L/rust_online/online_ant_colony
4,T12,scen_2000_120,146276,138488,8,online_ant_colony,51.613940,8.700000,3.822226,179682.546499,0,20,7.078810,20,19.055681,371351.886771,T12_sine_IP_L_online_0_1113_1705_20/scen_2000_120_sine_IP_L/rust_online/online_ant_colony
5,T14,scen_2000_120,146276,137292,8,online_ant_colony,52.468527,5.297000,3.824658,174980.834506,0,20,1.479880,20,17.624823,368136.779768,T14_sine_IP_L_online_2_1115_1748_20/scen_2000_120_sine_IP_L/rust_online/online_ant_colony
6,T14,scen_2000_120,146276,141281,8,online_ant_colony,53.351880,6.017000,3.887784,173740.631469,0,10,1.478557,10,9.677081,372449.374591,T14_sine_IP_L_online_1_1115_1748_10/scen_2000_120_sine_IP_L/rust_online/online_ant_colony
7,T13,scen_2000_120,146276,141716,8,online_ant_colony,52.924764,5.619000,3.892909,176295.231993,0,20,1.514750,20,17.357874,374496.755360,T13_sine_IP_L_online_0_1113_1740_20/scen_2000_120_sine_IP_L/rust_online/online_ant_colony
8,T10,scen_2000_120,146276,142462,8,online_ant_colony,53.583901,5.552000,3.904392,173559.028766,0,20,1.601804,20,17.099625,373919.898435,T10_sine_IP_L_online_0_1113_2330_20/scen_2000_120_sine_IP_L/rust_online/online_ant_colony
9,T14,scen_2000_120,146276,145820,8,online_ant_colony,50.726023,18.932000,3.957276,190638.516308,0,1,0.859231,1,1.000000,386894.926619,T14_sine_IP_L_online_0_1115_1746_1/scen_2000_120_sine_IP_L/rust_online/online_ant_colony


## Ant colony

In general ACO should perform best with higher batch_sizes, because a single "ant" corresponds to a whole solution, i.e. assignments for all the requests in the batch.
Considering a higher batch_size allows more possibilities to be explored.

However there is a limit to how much can be gained, as flushing will still occur:
- Consider that requests follow a sinusoidal pattern, with valleys below the average load and peaks up to 2x above it.\
- Consider also that the scheduler when entering a new slot, forces the flush of the requests in the queue, even if they are less than `batch_size`.\
- Thus, setting a high batch size like 1000 will imply flushing often, even if the queue is not full.\

There is also another aspect. ACO has a configurable amount of ants and iterations, which affects how many solutions it explores.\
For these experiments we stick to a maximum of `n_ants = 30` and `n_iterations = 10`, which limits the search effort to 400 solutions.\
(Note that low batch sizes impose a cap to explore fewer solutions)

Tests using `n_ants = 80` and `n_iterations = 30` bring the search effort to 2400 solutions, improving the quality but resulting in very high overhead time-wise (on average 12 seconds for a 3000 request batch)



### How many ants and how many iterations?
- T11: `n_ants = 30` and `n_iterations = 10`
- T12: `n_ants = 25` and `n_iterations = 5`
- T13: `n_ants = 10` and `n_iterations = 3`

We evaluate the performance with these configurations.


NOTE:
TODO: re-exec this. Even on a single scenario would be okay.
The following result has not been saved.
<pre style="font-size: 0.8em; line-height: 1.15;">
Wrote online-strategy output to /home/francis/unipi/carbonshift/carbonshift/tests/battery/results/T01_sine_IP_L_online_0_1112_2244_20/scen_1500_92_sine_IP_L/rust_online.
    [rust/online_ant_colony] N=1: solver_ms=0.2, carbon=125929.827, error=3.874, saving=121812.752 gCO2 (49.2%), elapsed=11.7s
    [rust/online_ant_colony] N=8: solver_ms=29.3, carbon=119021.852, error=3.800, saving=123768.986 gCO2 (51.0%), elapsed=21.7s
    [rust/online_ant_colony] N=100: solver_ms=1526.9, carbon=123350.889, error=3.774, saving=118449.686 gCO2 (49.0%), elapsed=131.4s
    [rust/online_ant_colony] N=250: solver_ms=2509.9, carbon=144624.407, error=3.775, saving=102612.044 gCO2 (41.5%), elapsed=191.8s
    [rust/online_ant_colony] N=500: solver_ms=3965.6, carbon=138590.402, error=3.924, saving=122214.981 gCO2 (46.9%), elapsed=337.5s
    [rust/online_ant_colony] N=1000: solver_ms=7042.6, carbon=125485.498, error=3.953, saving=135834.553 gCO2 (52.0%), elapsed=604.3s
    [rust/online_ant_colony] N=1500: solver_ms=8477.2, carbon=100413.235, error=3.956, saving=161089.357 gCO2 (61.6%), elapsed=716.6s
    [rust/online_ant_colony] N=2000: solver_ms=9893.8, carbon=90974.798, error=3.926, saving=163097.141 gCO2 (64.2%), elapsed=816.5s
    [rust/online_ant_colony] N=3000: solver_ms=12335.8, carbon=101097.899, error=3.920, saving=151528.065 gCO2 (60.0%), elapsed=900.1s
  Scenario scen_1500_92_sine_IP_L completed in 3759.0s
</pre>

In [68]:
# T11: n_ants = 30 and n_iterations = 10
# T12: n_ants = 25 and n_iterations = 5
# T13: n_ants = 10 and n_iterations = 3

aco_comparison = compare_metrics_wide(
    df_all,
    row_keys=["scenario_id", "batch"],
    comparison_column="run_id",
    comparison_values=["T11", "T12", "T13"],
    metric_sources=comparison_metric_sources,
    filters={"policy": "online_ant_colony"},
)
display_by_scenario(aco_comparison)

,scenario_id,batch,elapsed_s_T11,elapsed_s_T12,elapsed_s_T13,carb_saving_T11,carb_saving_T12,carb_saving_T13,error_T11,error_T12,error_T13,solver_ms_T11,solver_ms_T12,solver_ms_T13,carb_cost_T11,carb_cost_T12,carb_cost_T13
0,scen_1500_92_sine_IP_L,1,10.789000,10.883000,10.731000,50.014356,45.990891,49.600202,3.855406,3.948111,3.856697,0.203392,0.198371,0.129207,126767.856345,140970.605840,128263.518752
1,scen_1500_92_sine_IP_L,8,13.478000,7.738000,4.671000,51.733883,52.461074,50.729540,3.814236,3.811527,3.894034,17.697190,8.737613,1.988514,120577.549121,116775.046236,126354.587215
2,scen_1500_92_sine_IP_L,100,17.228000,8.206000,3.439000,46.631648,48.405255,43.080134,3.826755,3.849947,3.873754,187.087667,82.647214,26.849093,130675.410853,133372.013010,148169.545996
3,scen_1500_92_sine_IP_L,250,24.033000,11.189000,3.821000,50.249194,41.123477,46.163233,3.841851,3.816721,3.860303,301.927220,130.049742,37.178033,124204.846831,145838.600272,139770.363963
4,scen_1500_92_sine_IP_L,500,40.369000,17.676000,5.007000,48.217527,42.658382,49.428921,3.908818,3.904682,3.925550,461.344065,199.120060,48.634906,127730.393339,149544.961315,132137.169518
5,scen_1500_92_sine_IP_L,1000,69.391000,31.414000,8.677000,55.924906,55.037556,51.362764,3.952570,3.851478,3.913738,799.003802,351.864853,88.810870,115219.419362,116309.289391,126802.383545
6,scen_1500_92_sine_IP_L,1500,91.888000,39.693000,10.189000,53.979851,52.307930,55.343223,3.930173,3.873350,3.931239,1065.185891,455.883012,109.244569,120106.722039,124116.094506,114673.925931
7,scen_1500_92_sine_IP_L,2000,109.521000,44.949000,11.482000,51.535785,54.119839,56.157408,3.916544,3.894629,3.937035,1306.406176,528.730395,126.403527,126169.870054,119724.917987,114556.689313
8,scen_1500_92_sine_IP_L,3000,112.620000,46.871000,12.135000,58.473147,55.012063,58.644245,3.946307,3.895823,3.955543,1471.810280,618.169064,153.352468,108543.368292,117408.561675,108177.951947


,scenario_id,batch,elapsed_s_T11,elapsed_s_T12,elapsed_s_T13,carb_saving_T11,carb_saving_T12,carb_saving_T13,error_T11,error_T12,error_T13,solver_ms_T11,solver_ms_T12,solver_ms_T13,carb_cost_T11,carb_cost_T12,carb_cost_T13
0,scen_1800_230_sine_IP_L,1,13.868000,14.082000,13.770000,50.005038,53.905672,56.312952,3.933058,3.904853,3.850430,0.202402,0.189164,0.126541,159952.859676,142784.785082,131952.966157
1,scen_1800_230_sine_IP_L,8,16.660000,8.134000,5.541000,52.419843,44.635777,51.857080,3.891955,3.851988,3.902749,18.364440,7.583565,1.929687,147317.634423,169963.025151,151368.959848
2,scen_1800_230_sine_IP_L,100,20.660000,9.251000,3.838000,37.993930,50.122899,46.267476,3.865568,3.922638,3.858604,200.771973,85.039239,28.451314,196744.519665,159083.153241,170401.350561
3,scen_1800_230_sine_IP_L,250,26.436000,12.421000,4.127000,50.483644,51.767797,37.293789,3.808267,3.976258,3.895300,334.253287,151.329704,40.147144,146612.220303,154563.721730,200112.943574
4,scen_1800_230_sine_IP_L,500,40.196000,18.367000,5.106000,49.524519,48.324695,43.353154,3.872874,3.909027,3.885666,470.124065,208.701137,50.300734,157548.307038,164748.996702,180404.251548
5,scen_1800_230_sine_IP_L,1000,69.551000,33.009000,8.547000,60.749403,56.419616,49.177020,3.956956,3.902156,3.940242,774.156607,359.773376,84.721933,125707.116534,138745.597521,162446.627274
6,scen_1800_230_sine_IP_L,1500,98.375000,42.743000,10.660000,60.362392,56.859640,54.035271,3.960600,3.901080,3.849560,1125.481719,481.539552,112.046448,127008.148321,137498.109355,136716.698574
7,scen_1800_230_sine_IP_L,2000,123.051000,49.993000,12.065000,54.385636,54.018109,54.931429,3.893782,3.856858,3.937096,1421.555332,579.466141,130.723255,137574.611936,145352.426546,140383.496278
8,scen_1800_230_sine_IP_L,3000,137.179000,55.909000,14.602000,55.886940,54.113099,57.668505,3.924785,3.854833,3.912808,1788.237593,704.594666,178.338879,137548.371878,145061.683052,131815.834253


,scenario_id,batch,elapsed_s_T11,elapsed_s_T12,elapsed_s_T13,carb_saving_T11,carb_saving_T12,carb_saving_T13,error_T11,error_T12,error_T13,solver_ms_T11,solver_ms_T12,solver_ms_T13,carb_cost_T11,carb_cost_T12,carb_cost_T13
0,scen_2000_120_sine_IP_L,1,16.824000,16.995000,17.392000,52.190349,49.419405,44.449841,3.938931,3.949453,3.930397,0.201145,0.185441,0.132268,184622.136624,195645.537496,214091.301995
1,scen_2000_120_sine_IP_L,8,16.981000,8.700000,5.619000,54.574711,51.613940,52.924764,3.818417,3.822226,3.892909,16.470374,7.078810,1.514750,167100.777838,179682.546499,176295.231993
2,scen_2000_120_sine_IP_L,100,22.257000,10.202000,3.681000,42.607594,46.576333,50.912305,3.901538,3.825464,3.875176,199.834695,86.664910,24.448163,220167.759198,188884.428067,187293.860930
3,scen_2000_120_sine_IP_L,250,28.054000,12.559000,4.766000,50.133365,44.968206,49.853015,3.866095,3.910027,3.859377,353.429071,150.305771,46.134168,188554.594532,209807.448763,190208.183635
4,scen_2000_120_sine_IP_L,500,43.721000,18.548000,5.596000,48.253621,55.288263,44.338733,3.857616,3.848076,3.918662,527.575467,213.244189,56.583358,192900.775873,170311.802057,215435.405463
5,scen_2000_120_sine_IP_L,1000,73.745000,32.685000,8.499000,52.762788,47.963259,49.671894,3.943121,3.906611,3.834022,804.188190,354.696297,85.207064,182523.941008,200605.916602,181186.738039
6,scen_2000_120_sine_IP_L,1500,101.689000,41.968000,11.383000,61.506933,57.437542,57.479647,3.967069,3.958269,3.826050,1134.631304,460.561948,118.355995,149036.444989,164903.652973,155126.508822
7,scen_2000_120_sine_IP_L,2000,127.093000,51.411000,13.642000,55.845632,53.177561,58.436050,3.926252,3.943700,3.833259,1440.850199,570.312501,141.814992,170714.264281,181304.436422,149912.123587
8,scen_2000_120_sine_IP_L,3000,152.052000,61.870000,15.650000,54.289457,52.891158,55.167410,3.848167,3.964716,3.833064,1836.835313,748.585617,179.976197,173723.573595,182468.456022,161552.963258


A higher number of ants and of iterations does not seem to imply more saving. The overhead instead, is significative.\
Hence, we stick to a low number of ants/iterations.

We may stick to batch size = 8

## Comparing ACO, MBA, Hazard and singleton_best


### scen_1500_92

In [69]:
df_on = df[(df["run_id"] == "T10")]
pd.set_option("display.max_rows", 28)
# display(df_on.sort_values(by=SORTING_CRITERIA))
df_92 = df_on[df_on["scenario_id"] == "scen_1500_92"]
display_by_scenario(df_92.sort_values(by=SORTING_CRITERIA))

,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T10,scen_1500_92,105723,104045,8,online_ant_colony,49.262817,4.622000,3.891874,130133.339424,0,20,1.994317,20,17.118457,256485.147770,T10_sine_IP_L_online_0_1113_2330_20/scen_1500_92_sine_IP_L/rust_online/online_ant_colony
1,T10,scen_1500_92,105723,98454,1,online_bandit,50.738682,10.195000,3.838620,120190.695148,0,20,0.006489,20,18.036359,243985.948614,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1500_92_sine_IP_L/rust_online/online_bandit
2,T10,scen_1500_92,105723,101006,8,online_bandit,45.964754,3.284000,3.813214,136610.555536,0,20,0.046308,20,11.425054,252817.496619,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1500_92_sine_IP_L/rust_online/online_bandit
3,T10,scen_1500_92,105723,102421,100,online_bandit,48.372054,1.922000,3.814525,132146.754816,0,20,0.814600,20,8.730282,255959.735979,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1500_92_sine_IP_L/rust_online/online_bandit
4,T10,scen_1500_92,105723,104956,250,online_bandit,45.897563,1.785000,3.804523,140930.202615,0,20,1.791592,13,5.180909,260487.716952,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1500_92_sine_IP_L/rust_online/online_bandit
5,T10,scen_1500_92,105723,105197,500,online_bandit,51.142744,1.723000,3.849159,127421.066345,0,20,3.252258,7,2.974822,260802.746692,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1500_92_sine_IP_L/rust_online/online_bandit
6,T10,scen_1500_92,105723,99756,1000,online_bandit,47.268702,1.689000,3.848340,130222.117208,0,20,5.253364,4,1.600378,246954.129920,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1500_92_sine_IP_L/rust_online/online_bandit
7,T10,scen_1500_92,105723,98106,1500,online_bandit,47.521237,1.708000,3.816382,127877.624451,0,20,5.835930,3,1.242068,243674.995680,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1500_92_sine_IP_L/rust_online/online_bandit
8,T10,scen_1500_92,105723,98051,2000,online_bandit,48.809732,1.822000,3.797004,125097.645641,0,20,7.125229,2,1.088862,244377.792073,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1500_92_sine_IP_L/rust_online/online_bandit
9,T10,scen_1500_92,105723,97707,3000,online_bandit,48.713147,1.830000,3.794099,124234.925164,0,20,7.558089,2,1.002694,242235.422338,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1500_92_sine_IP_L/rust_online/online_bandit


### scen_1800_230

In [70]:
# pd.set_option("display.max_rows", 28)
# display(df_on.sort_values(by=SORTING_CRITERIA))
df_230 = df_on[df_on["scenario_id"] == "scen_1800_230"]
display_by_scenario(df_230.sort_values(by=SORTING_CRITERIA))

,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T10,scen_1800_230,128171,126335,8,online_ant_colony,53.253764,5.232000,3.908715,147127.201299,0,20,1.763743,20,17.346429,314735.929042,T10_sine_IP_L_online_0_1113_2330_20/scen_1800_230_sine_IP_L/rust_online/online_ant_colony
1,T10,scen_1800_230,128171,122211,1,online_bandit,55.237031,13.341000,3.877965,135672.284588,0,20,0.006534,20,18.289354,303090.451744,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1800_230_sine_IP_L/rust_online/online_bandit
2,T10,scen_1800_230,128171,123727,8,online_bandit,40.514440,3.642000,3.874963,186486.733627,0,20,0.043087,20,12.818874,313499.166881,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1800_230_sine_IP_L/rust_online/online_bandit
3,T10,scen_1800_230,128171,122196,100,online_bandit,48.048111,2.055000,3.796503,160697.003328,0,20,0.692522,19,8.328768,309318.885405,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1800_230_sine_IP_L/rust_online/online_bandit
4,T10,scen_1800_230,128171,123177,250,online_bandit,40.759673,1.841000,3.795047,184879.433411,0,20,1.877178,14,5.860526,312083.751138,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1800_230_sine_IP_L/rust_online/online_bandit
5,T10,scen_1800_230,128171,126231,500,online_bandit,42.721629,1.730000,3.788253,182082.342828,0,20,2.685542,8,3.455899,317890.223285,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1800_230_sine_IP_L/rust_online/online_bandit
6,T10,scen_1800_230,128171,120032,1000,online_bandit,51.510847,1.755000,3.847245,146005.702476,0,20,4.628149,4,1.925071,301110.025717,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1800_230_sine_IP_L/rust_online/online_bandit
7,T10,scen_1800_230,128171,119958,1500,online_bandit,44.659434,1.673000,3.789472,166530.664609,0,20,5.382949,3,1.401294,300919.698854,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1800_230_sine_IP_L/rust_online/online_bandit
8,T10,scen_1800_230,128171,122633,2000,online_bandit,45.728403,1.752000,3.799528,168462.820959,0,20,6.126186,3,1.165153,310406.971297,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1800_230_sine_IP_L/rust_online/online_bandit
9,T10,scen_1800_230,128171,122621,3000,online_bandit,43.113285,1.845000,3.808503,176502.513875,0,20,8.334732,2,1.019997,310270.182577,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1800_230_sine_IP_L/rust_online/online_bandit


### scen_2000_120

In [71]:
df_on = df[(df["run_id"] == "T10")]
pd.set_option("display.max_rows", 28)
# display(df_on.sort_values(by=SORTING_CRITERIA))
df_120 = df_on[df_on["scenario_id"] == "scen_2000_120"]
display_by_scenario(df_120.sort_values(by=SORTING_CRITERIA))

,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T10,scen_2000_120,146276,142462,8,online_ant_colony,53.583901,5.552000,3.904392,173559.028766,0,20,1.601804,20,17.099625,373919.898435,T10_sine_IP_L_online_0_1113_2330_20/scen_2000_120_sine_IP_L/rust_online/online_ant_colony
1,T10,scen_2000_120,146276,145265,1,online_bandit,51.925500,16.549000,3.869463,185596.104366,0,20,0.006463,20,18.446621,386059.350964,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2000_120_sine_IP_L/rust_online/online_bandit
2,T10,scen_2000_120,146276,138929,8,online_bandit,51.315932,4.380000,3.877556,177929.731388,0,20,0.042221,20,12.948549,365478.355101,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2000_120_sine_IP_L/rust_online/online_bandit
3,T10,scen_2000_120,146276,138998,100,online_bandit,48.259794,2.021000,3.781421,192854.694965,0,20,0.499584,20,8.322044,372736.616693,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2000_120_sine_IP_L/rust_online/online_bandit
4,T10,scen_2000_120,146276,143140,250,online_bandit,37.031215,1.793000,3.816840,240357.985205,0,20,1.499518,17,6.266653,381709.741337,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2000_120_sine_IP_L/rust_online/online_bandit
5,T10,scen_2000_120,146276,142396,500,online_bandit,36.337830,1.758000,3.812168,242603.580934,0,20,2.679694,9,3.949498,381079.661455,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2000_120_sine_IP_L/rust_online/online_bandit
6,T10,scen_2000_120,146276,139749,1000,online_bandit,43.361832,1.735000,3.820278,212104.098698,0,20,4.082268,5,2.110459,374489.686521,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2000_120_sine_IP_L/rust_online/online_bandit
7,T10,scen_2000_120,146276,136330,1500,online_bandit,49.292807,1.773000,3.801419,184724.473938,0,20,5.310431,4,1.528262,364296.391029,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2000_120_sine_IP_L/rust_online/online_bandit
8,T10,scen_2000_120,146276,137666,2000,online_bandit,43.531604,1.962000,3.808293,208065.016559,0,20,5.928615,3,1.242900,368462.773573,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2000_120_sine_IP_L/rust_online/online_bandit
9,T10,scen_2000_120,146276,135479,3000,online_bandit,48.628951,1.903000,3.776655,186615.468010,0,20,6.549335,5,1.095258,363269.722162,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2000_120_sine_IP_L/rust_online/online_bandit


### singleton_best is our winner

Singleton_best outperforms others.

High batch_size may yield slightly faster results, but make sense only with very high load. They do not seem to imply better savings (actually worse, typically).

We may consider batch_size = 8 the best fit, for both speed and achieved saving.


Singleton_best gets us the best carbon savings.
Hazard almost gets there, but singleton_best  can make fine grained adjustments, while hazard simply blindly follows probabilities.
Bandit and ACO lag begind by 25%, which is considerable.

## T14 - Varying parallelism for our tuned policies

In [72]:
comparison_parallelisms = [1, 10, 20]
t14_parallelism_comparison = compare_metrics_wide(
    df_all,
    row_keys=["scenario_id", "batch", "policy"],
    comparison_column="paral",
    comparison_values=comparison_parallelisms,
    metric_sources=comparison_metric_sources,
    filters={"run_id": "T14"},
    labels={1: "P1", 10: "P10", 20: "P20"},
    key_renames={"batch": "batch_size"},
)
display_by_scenario(t14_parallelism_comparison)

,scenario_id,batch_size,policy,elapsed_s_P1,elapsed_s_P10,elapsed_s_P20,carb_saving_P1,carb_saving_P10,carb_saving_P20,error_P1,error_P10,error_P20,solver_ms_P1,solver_ms_P10,solver_ms_P20,carb_cost_P1,carb_cost_P10,carb_cost_P20
0,scen_1500_92_sine_IP_L,8,online_ant_colony,13.656000,5.000000,4.835000,52.516907,50.921240,50.775329,3.989975,3.879190,3.841224,0.838445,1.801890,2.078546,123864.794353,123869.934946,125465.428783
1,scen_1500_92_sine_IP_L,8,online_bandit,3.029000,3.119000,3.160000,45.948229,43.446332,43.658000,3.791343,3.800836,3.814286,0.035973,0.040164,0.040922,132646.170840,144224.780121,143246.759496
2,scen_1500_92_sine_IP_L,8,online_hazard,2.919000,3.097000,3.398000,74.834967,74.836624,74.828425,3.999454,3.999195,3.999364,0.010020,0.010722,0.011788,63033.745666,63020.481164,63036.431669
3,scen_1500_92_sine_IP_L,8,online_singleton_best,2.763000,2.990000,3.328000,78.110563,77.910686,77.661753,3.997761,3.991847,3.986636,0.002654,0.003769,0.003896,52177.912780,52691.989929,53306.598510


,scenario_id,batch_size,policy,elapsed_s_P1,elapsed_s_P10,elapsed_s_P20,carb_saving_P1,carb_saving_P10,carb_saving_P20,error_P1,error_P10,error_P20,solver_ms_P1,solver_ms_P10,solver_ms_P20,carb_cost_P1,carb_cost_P10,carb_cost_P20
0,scen_1800_230_sine_IP_L,8,online_ant_colony,15.394000,5.660000,4.786000,55.613596,54.913248,51.473263,3.968086,3.853355,3.863037,0.777045,1.665819,1.580342,141260.393367,136698.920400,151981.922185
1,scen_1800_230_sine_IP_L,8,online_bandit,3.601000,3.654000,3.498000,55.290682,50.907993,46.034676,3.873685,3.820424,3.821213,0.037350,0.041560,0.039188,135521.928923,146117.866285,170482.492107
2,scen_1800_230_sine_IP_L,8,online_hazard,3.220000,3.576000,3.532000,74.125593,74.139354,74.109858,3.999804,3.999563,3.999400,0.009538,0.010406,0.010281,79761.335812,79699.936368,79781.496095
3,scen_1800_230_sine_IP_L,8,online_singleton_best,3.228000,3.426000,3.475000,77.281806,76.885055,76.877865,3.995254,3.987548,3.984090,0.002795,0.003328,0.003616,66743.368637,67961.393744,68009.548032


,scenario_id,batch_size,policy,elapsed_s_P1,elapsed_s_P10,elapsed_s_P20,carb_saving_P1,carb_saving_P10,carb_saving_P20,error_P1,error_P10,error_P20,solver_ms_P1,solver_ms_P10,solver_ms_P20,carb_cost_P1,carb_cost_P10,carb_cost_P20
0,scen_2000_120_sine_IP_L,8,online_ant_colony,18.932000,6.017000,5.297000,50.726023,53.351880,52.468527,3.957276,3.887784,3.824658,0.859231,1.478557,1.479880,190638.516308,173740.631469,174980.834506
1,scen_2000_120_sine_IP_L,8,online_bandit,4.120000,4.172000,3.868000,50.489591,49.042546,50.024369,3.780361,3.810636,3.912212,0.035929,0.042619,0.038120,180006.911491,191979.010948,193540.724020
2,scen_2000_120_sine_IP_L,8,online_hazard,3.471000,3.943000,3.993000,74.844695,74.822626,74.786920,3.998468,3.998619,3.998080,0.008903,0.010219,0.010403,93290.712760,93362.557145,93507.230929
3,scen_2000_120_sine_IP_L,8,online_singleton_best,3.628000,4.015000,3.891000,78.101955,77.808173,77.549438,3.998739,3.991180,3.983640,0.002630,0.003583,0.003400,77181.720319,78259.825610,79246.893435


### Parallelism is beneficial for ACO, not others
Parallelism helps speed up ACO runs, not other policies. It neglectably degrades performance.

# TEST 30 | In-Phase comparison

In [73]:
# df_30 = df_all[(df_all["path"].str.contains("T30"))]
df_30 = df_all[df_all["path"].str.contains("^T30")]

pd.set_option("display.max_rows", 28)
# display(df_30.sort_values(by=SORTING_CRITERIA))
df_120 = df_30[df_30["scenario_id"].str.contains("scen_2000_120")]
display_by_scenario(df_120.sort_values(by=SORTING_CRITERIA))

df_92 = df_30[df_30["scenario_id"].str.contains("scen_1500_92")]
display_by_scenario(df_92.sort_values(by=SORTING_CRITERIA))

df_230 = df_30[df_30["scenario_id"].str.contains("scen_1800_230")]
display_by_scenario(df_230.sort_values(by=SORTING_CRITERIA))


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T30,scen_2000_120_sine_IP_L,146276,132740,1,dp,78.029072,16.635000,3.999680,77419.696489,240,20,0.013953,20,18.759955,352373.359083,T30_sine_IP_L_dp_0_1115_2246_20/scen_2000_120_sine_IP_L/rust_dp
1,T30,scen_2000_120_sine_IP_L,146276,137610,0,offline_ant_colony,64.664853,14.832000,3.869722,129399.188488,0,20,14761.904938,0,0.000000,366205.321878,T30_sine_IP_L_offline_0_1115_2250_20/scen_2000_120_sine_IP_L/rust_offline/strategy_ant_colony
2,T30,scen_2000_120_sine_IP_L,146276,146276,0,offline_arrival_slot,44.301113,0.051000,2.491540,215839.952888,0,20,3.698213,0,0.000000,387512.144628,T30_sine_IP_L_offline_0_1115_2250_20/scen_2000_120_sine_IP_L/rust_offline/strategy_arrival_slot
3,T30,scen_2000_120_sine_IP_L,146276,144639,0,offline_bandit,55.496760,0.561000,3.892795,171334.083622,0,20,500.028090,0,0.000000,384992.378728,T30_sine_IP_L_offline_0_1115_2250_20/scen_2000_120_sine_IP_L/rust_offline/strategy_bandit
4,T30,scen_2000_120_sine_IP_L,146276,139156,0,offline_hazard,74.789949,0.080000,3.998067,93544.318399,0,20,27.847416,0,0.000000,371059.619989,T30_sine_IP_L_offline_0_1115_2250_20/scen_2000_120_sine_IP_L/rust_offline/strategy_hazard
5,T30,scen_2000_120_sine_IP_L,146276,140042,0,offline_random_slot,51.796801,0.055000,2.501625,180008.044842,0,20,4.597615,0,0.000000,373435.886444,T30_sine_IP_L_offline_0_1115_2250_20/scen_2000_120_sine_IP_L/rust_offline/strategy_random_slot
6,T30,scen_2000_120_sine_IP_L,146276,132790,0,offline_singleton_best,78.066716,0.087000,3.997044,77328.378454,0,20,19.609069,0,0.000000,352561.794108,T30_sine_IP_L_offline_0_1115_2250_20/scen_2000_120_sine_IP_L/rust_offline/strategy_singleton_best
7,T30,scen_2000_120_sine_IP_L,146276,132746,0,offline_singleton_best_accurate,27.194702,0.067000,0.000000,256612.834892,0,20,15.280483,0,0.000000,352464.505006,T30_sine_IP_L_offline_0_1115_2250_20/scen_2000_120_sine_IP_L/rust_offline/strategy_singleton_best_accurate
8,T30,scen_2000_120_sine_IP_L,146276,142228,8,online_ant_colony,56.044108,5.403000,3.911642,165680.782355,0,20,1.550473,20,17.517284,376925.080402,T30_sine_IP_L_online_0_1115_2247_20/scen_2000_120_sine_IP_L/rust_online/online_ant_colony
9,T30,scen_2000_120_sine_IP_L,146276,146276,8,online_arrival_slot,44.518090,3.926000,2.504666,214999.138615,1,20,0.000988,20,13.428000,387512.144628,T30_sine_IP_L_perrequest_0_1115_2248_20/scen_2000_120_sine_IP_L/rust_online/online_arrival_slot


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T30,scen_1500_92_sine_IP_L,105723,95985,1,dp,78.000587,10.552000,3.997578,52425.262163,285,20,0.014310,20,18.169708,238303.004233,T30_sine_IP_L_dp_0_1115_2246_20/scen_1500_92_sine_IP_L/rust_dp
1,T30,scen_1500_92_sine_IP_L,105723,99608,0,offline_ant_colony,75.438568,10.975000,3.991070,61011.624355,0,20,10933.735461,0,0.000000,248404.186800,T30_sine_IP_L_offline_0_1115_2250_20/scen_1500_92_sine_IP_L/rust_offline/strategy_ant_colony
2,T30,scen_1500_92_sine_IP_L,105723,105723,0,offline_arrival_slot,44.558623,0.038000,2.498037,145064.056011,1,20,2.497702,0,0.000000,261653.054075,T30_sine_IP_L_offline_0_1115_2250_20/scen_1500_92_sine_IP_L/rust_offline/strategy_arrival_slot
3,T30,scen_1500_92_sine_IP_L,105723,105388,0,offline_bandit,55.383008,0.400000,3.898784,116541.776787,0,20,360.731398,0,0.000000,261204.918797,T30_sine_IP_L_offline_0_1115_2250_20/scen_1500_92_sine_IP_L/rust_offline/strategy_bandit
4,T30,scen_1500_92_sine_IP_L,105723,100680,0,offline_hazard,74.896912,0.056000,4.003303,62938.751509,0,20,19.991312,0,0.000000,250721.150889,T30_sine_IP_L_offline_0_1115_2250_20/scen_1500_92_sine_IP_L/rust_offline/strategy_hazard
5,T30,scen_1500_92_sine_IP_L,105723,101130,0,offline_random_slot,51.785395,0.040000,2.505068,121384.914377,1,20,3.303123,0,0.000000,251759.636403,T30_sine_IP_L_offline_0_1115_2250_20/scen_1500_92_sine_IP_L/rust_offline/strategy_random_slot
6,T30,scen_1500_92_sine_IP_L,105723,96001,0,offline_singleton_best,78.128682,0.063000,3.998188,52097.614477,0,20,14.292306,0,0.000000,238200.620990,T30_sine_IP_L_offline_0_1115_2250_20/scen_1500_92_sine_IP_L/rust_offline/strategy_singleton_best
7,T30,scen_1500_92_sine_IP_L,105723,95982,0,offline_singleton_best_accurate,27.285229,0.048000,0.000000,173169.572013,0,20,11.068168,0,0.000000,238149.099099,T30_sine_IP_L_offline_0_1115_2250_20/scen_1500_92_sine_IP_L/rust_offline/strategy_singleton_best_accurate
8,T30,scen_1500_92_sine_IP_L,105723,101664,8,online_ant_colony,48.310623,4.596000,3.840543,131321.406215,0,20,1.921165,20,17.120783,254058.788474,T30_sine_IP_L_online_0_1115_2247_20/scen_1500_92_sine_IP_L/rust_online/online_ant_colony
9,T30,scen_1500_92_sine_IP_L,105723,105723,8,online_arrival_slot,44.481710,2.959000,2.503098,145265.300144,1,20,0.001048,20,12.065065,261653.054075,T30_sine_IP_L_perrequest_0_1115_2248_20/scen_1500_92_sine_IP_L/rust_online/online_arrival_slot


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T30,scen_1800_230_sine_IP_L,128171,117217,1,dp,77.280702,13.277000,3.999143,66729.132588,219,20,0.013675,20,18.374290,293711.239770,T30_sine_IP_L_dp_0_1115_2246_20/scen_1800_230_sine_IP_L/rust_dp
1,T30,scen_1800_230_sine_IP_L,128171,122671,0,offline_ant_colony,61.176406,12.869000,3.885331,119457.152555,0,20,12812.388593,0,0.000000,307692.152553,T30_sine_IP_L_offline_0_1115_2250_20/scen_1800_230_sine_IP_L/rust_offline/strategy_ant_colony
2,T30,scen_1800_230_sine_IP_L,128171,128171,0,offline_arrival_slot,44.484203,0.047000,2.498615,177912.194806,0,20,3.281001,0,0.000000,320471.297784,T30_sine_IP_L_offline_0_1115_2250_20/scen_1800_230_sine_IP_L/rust_offline/strategy_arrival_slot
3,T30,scen_1800_230_sine_IP_L,128171,124127,0,offline_bandit,49.082860,0.488000,3.933008,158483.422149,0,20,439.946461,0,0.000000,311257.512422,T30_sine_IP_L_offline_0_1115_2250_20/scen_1800_230_sine_IP_L/rust_offline/strategy_bandit
4,T30,scen_1800_230_sine_IP_L,128171,122480,0,offline_hazard,74.129550,0.076000,3.995367,79694.814285,0,20,24.605884,0,0.000000,308053.456934,T30_sine_IP_L_offline_0_1115_2250_20/scen_1800_230_sine_IP_L/rust_offline/strategy_hazard
5,T30,scen_1800_230_sine_IP_L,128171,123100,0,offline_random_slot,50.113643,0.047000,2.493339,154627.079324,1,20,3.943014,0,0.000000,309958.653246,T30_sine_IP_L_offline_0_1115_2250_20/scen_1800_230_sine_IP_L/rust_offline/strategy_random_slot
6,T30,scen_1800_230_sine_IP_L,128171,117263,0,offline_singleton_best,77.321231,0.081000,3.996956,66621.794793,0,20,16.812199,0,0.000000,293762.832203,T30_sine_IP_L_offline_0_1115_2250_20/scen_1800_230_sine_IP_L/rust_offline/strategy_singleton_best
7,T30,scen_1800_230_sine_IP_L,128171,117218,0,offline_singleton_best_accurate,24.794494,0.057000,0.000000,220870.964860,0,20,13.216180,0,0.000000,293689.887632,T30_sine_IP_L_offline_0_1115_2250_20/scen_1800_230_sine_IP_L/rust_offline/strategy_singleton_best_accurate
8,T30,scen_1800_230_sine_IP_L,128171,123784,8,online_ant_colony,55.313981,4.163000,3.919691,138821.030116,0,20,1.318299,20,17.474120,310658.752360,T30_sine_IP_L_online_0_1115_2247_20/scen_1800_230_sine_IP_L/rust_online/online_ant_colony
9,T30,scen_1800_230_sine_IP_L,128171,128171,8,online_arrival_slot,44.521083,3.244000,2.504389,177794.006493,1,20,0.000890,20,13.323440,320471.297784,T30_sine_IP_L_perrequest_0_1115_2248_20/scen_1800_230_sine_IP_L/rust_online/online_arrival_slot


In [74]:
def top5_by_carbon_saving(data, scenario_id, top_n=5, worst_n=False):
    """Return the highest-carbon-saving rows for one scenario."""
    required_columns = {"scenario_id", "carb_saving"}
    missing_columns = required_columns - set(data.columns)
    if missing_columns:
        raise ValueError(f"Dataframe is missing required columns: {sorted(missing_columns)}")
    if top_n < 1:
        raise ValueError("top_n must be at least 1")

    scenario_rows = data.loc[data["scenario_id"].eq(scenario_id)].copy()
    if scenario_rows.empty:
        raise ValueError(f"No rows found for scenario_id={scenario_id!r}")

    scenario_rows["carb_saving"] = pd.to_numeric(
        scenario_rows["carb_saving"], errors="raise"
    )
    if scenario_rows["carb_saving"].isna().any():
        raise ValueError(f"Scenario {scenario_id!r} has missing carbon-saving values")

    return (
        scenario_rows.sort_values(
            "carb_saving", ascending=worst_n, kind="stable"
        )
        .head(top_n)
        .reset_index(drop=True)
    )


def top5_by_carbon_saving_all_scenarios(data, top_n=5,worst_n=False):
    """Return up to top_n carbon-saving rows for each scenario in data."""
    if "scenario_id" not in data.columns:
        raise ValueError("Dataframe is missing required column: 'scenario_id'")

    scenario_ids = sorted(data["scenario_id"].dropna().unique())
    if not scenario_ids:
        raise ValueError("Dataframe contains no scenario IDs")

    return pd.concat(
        [top5_by_carbon_saving(data, scenario_id, top_n, worst_n) for scenario_id in scenario_ids],
        ignore_index=True,
    )


df_30_top5_by_scenario = top5_by_carbon_saving_all_scenarios(df_30)
df_30_worst5_by_scenario = top5_by_carbon_saving_all_scenarios(df_30, worst_n=True)
display_by_scenario(df_30_top5_by_scenario, title="Top five per scenario by carbon saving")
display_by_scenario(df_30_worst5_by_scenario, title="Bottom five per scenario by carbon saving")

,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T30,scen_1500_92_sine_IP_L,105723,96001,0,offline_singleton_best,78.128682,0.063000,3.998188,52097.614477,0,20,14.292306,0,0.000000,238200.620990,T30_sine_IP_L_offline_0_1115_2250_20/scen_1500_92_sine_IP_L/rust_offline/strategy_singleton_best
1,T30,scen_1500_92_sine_IP_L,105723,95985,1,dp,78.000587,10.552000,3.997578,52425.262163,285,20,0.014310,20,18.169708,238303.004233,T30_sine_IP_L_dp_0_1115_2246_20/scen_1500_92_sine_IP_L/rust_dp
2,T30,scen_1500_92_sine_IP_L,105723,96127,8,online_singleton_best,77.722642,2.837000,3.987745,53158.009394,0,20,0.003178,20,12.910572,238619.005382,T30_sine_IP_L_perrequest_0_1115_2248_20/scen_1500_92_sine_IP_L/rust_online/online_singleton_best
3,T30,scen_1500_92_sine_IP_L,105723,99608,0,offline_ant_colony,75.438568,10.975000,3.991070,61011.624355,0,20,10933.735461,0,0.000000,248404.186800,T30_sine_IP_L_offline_0_1115_2250_20/scen_1500_92_sine_IP_L/rust_offline/strategy_ant_colony
4,T30,scen_1500_92_sine_IP_L,105723,100680,0,offline_hazard,74.896912,0.056000,4.003303,62938.751509,0,20,19.991312,0,0.000000,250721.150889,T30_sine_IP_L_offline_0_1115_2250_20/scen_1500_92_sine_IP_L/rust_offline/strategy_hazard


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T30,scen_1800_230_sine_IP_L,128171,117263,0,offline_singleton_best,77.321231,0.081000,3.996956,66621.794793,0,20,16.812199,0,0.000000,293762.832203,T30_sine_IP_L_offline_0_1115_2250_20/scen_1800_230_sine_IP_L/rust_offline/strategy_singleton_best
1,T30,scen_1800_230_sine_IP_L,128171,117217,1,dp,77.280702,13.277000,3.999143,66729.132588,219,20,0.013675,20,18.374290,293711.239770,T30_sine_IP_L_dp_0_1115_2246_20/scen_1800_230_sine_IP_L/rust_dp
2,T30,scen_1800_230_sine_IP_L,128171,117425,8,online_singleton_best,77.030875,3.561000,3.989589,67546.414181,0,20,0.003469,20,13.038400,294074.828284,T30_sine_IP_L_perrequest_0_1115_2248_20/scen_1800_230_sine_IP_L/rust_online/online_singleton_best
3,T30,scen_1800_230_sine_IP_L,128171,122430,8,online_hazard,74.139317,3.445000,3.999408,79682.019744,0,20,0.009600,20,13.257817,308120.321244,T30_sine_IP_L_perrequest_0_1115_2248_20/scen_1800_230_sine_IP_L/rust_online/online_hazard
4,T30,scen_1800_230_sine_IP_L,128171,122480,0,offline_hazard,74.129550,0.076000,3.995367,79694.814285,0,20,24.605884,0,0.000000,308053.456934,T30_sine_IP_L_offline_0_1115_2250_20/scen_1800_230_sine_IP_L/rust_offline/strategy_hazard


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T30,scen_2000_120_sine_IP_L,146276,132790,0,offline_singleton_best,78.066716,0.087000,3.997044,77328.378454,0,20,19.609069,0,0.000000,352561.794108,T30_sine_IP_L_offline_0_1115_2250_20/scen_2000_120_sine_IP_L/rust_offline/strategy_singleton_best
1,T30,scen_2000_120_sine_IP_L,146276,132740,1,dp,78.029072,16.635000,3.999680,77419.696489,240,20,0.013953,20,18.759955,352373.359083,T30_sine_IP_L_dp_0_1115_2246_20/scen_2000_120_sine_IP_L/rust_dp
2,T30,scen_2000_120_sine_IP_L,146276,132900,8,online_singleton_best,77.754345,3.677000,3.990011,78475.332985,0,20,0.003163,20,13.692723,352767.021063,T30_sine_IP_L_perrequest_0_1115_2248_20/scen_2000_120_sine_IP_L/rust_online/online_singleton_best
3,T30,scen_2000_120_sine_IP_L,146276,139104,8,online_hazard,74.814500,3.858000,3.998178,93408.747531,0,20,0.010115,20,14.000760,370883.041499,T30_sine_IP_L_perrequest_0_1115_2248_20/scen_2000_120_sine_IP_L/rust_online/online_hazard
4,T30,scen_2000_120_sine_IP_L,146276,139156,0,offline_hazard,74.789949,0.080000,3.998067,93544.318399,0,20,27.847416,0,0.000000,371059.619989,T30_sine_IP_L_offline_0_1115_2250_20/scen_2000_120_sine_IP_L/rust_offline/strategy_hazard


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T30,scen_1500_92_sine_IP_L,105723,95982,0,offline_singleton_best_accurate,27.285229,0.048000,0.000000,173169.572013,0,20,11.068168,0,0.000000,238149.099099,T30_sine_IP_L_offline_0_1115_2250_20/scen_1500_92_sine_IP_L/rust_offline/strategy_singleton_best_accurate
1,T30,scen_1500_92_sine_IP_L,105723,105723,8,online_arrival_slot,44.481710,2.959000,2.503098,145265.300144,1,20,0.001048,20,12.065065,261653.054075,T30_sine_IP_L_perrequest_0_1115_2248_20/scen_1500_92_sine_IP_L/rust_online/online_arrival_slot
2,T30,scen_1500_92_sine_IP_L,105723,105723,0,offline_arrival_slot,44.558623,0.038000,2.498037,145064.056011,1,20,2.497702,0,0.000000,261653.054075,T30_sine_IP_L_offline_0_1115_2250_20/scen_1500_92_sine_IP_L/rust_offline/strategy_arrival_slot
3,T30,scen_1500_92_sine_IP_L,105723,101664,8,online_ant_colony,48.310623,4.596000,3.840543,131321.406215,0,20,1.921165,20,17.120783,254058.788474,T30_sine_IP_L_online_0_1115_2247_20/scen_1500_92_sine_IP_L/rust_online/online_ant_colony
4,T30,scen_1500_92_sine_IP_L,105723,101274,8,online_random_slot,51.730707,2.934000,2.496791,121715.062523,0,20,0.001091,20,12.647379,252158.372379,T30_sine_IP_L_perrequest_0_1115_2248_20/scen_1500_92_sine_IP_L/rust_online/online_random_slot


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T30,scen_1800_230_sine_IP_L,128171,117218,0,offline_singleton_best_accurate,24.794494,0.057000,0.000000,220870.964860,0,20,13.216180,0,0.000000,293689.887632,T30_sine_IP_L_offline_0_1115_2250_20/scen_1800_230_sine_IP_L/rust_offline/strategy_singleton_best_accurate
1,T30,scen_1800_230_sine_IP_L,128171,120842,8,online_bandit,44.132246,3.469000,3.803686,170780.749786,0,20,0.038342,20,12.595403,305687.517412,T30_sine_IP_L_perrequest_0_1115_2248_20/scen_1800_230_sine_IP_L/rust_online/online_bandit
2,T30,scen_1800_230_sine_IP_L,128171,128171,0,offline_arrival_slot,44.484203,0.047000,2.498615,177912.194806,0,20,3.281001,0,0.000000,320471.297784,T30_sine_IP_L_offline_0_1115_2250_20/scen_1800_230_sine_IP_L/rust_offline/strategy_arrival_slot
3,T30,scen_1800_230_sine_IP_L,128171,128171,8,online_arrival_slot,44.521083,3.244000,2.504389,177794.006493,1,20,0.000890,20,13.323440,320471.297784,T30_sine_IP_L_perrequest_0_1115_2248_20/scen_1800_230_sine_IP_L/rust_online/online_arrival_slot
4,T30,scen_1800_230_sine_IP_L,128171,124127,0,offline_bandit,49.082860,0.488000,3.933008,158483.422149,0,20,439.946461,0,0.000000,311257.512422,T30_sine_IP_L_offline_0_1115_2250_20/scen_1800_230_sine_IP_L/rust_offline/strategy_bandit


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T30,scen_2000_120_sine_IP_L,146276,132746,0,offline_singleton_best_accurate,27.194702,0.067000,0.000000,256612.834892,0,20,15.280483,0,0.000000,352464.505006,T30_sine_IP_L_offline_0_1115_2250_20/scen_2000_120_sine_IP_L/rust_offline/strategy_singleton_best_accurate
1,T30,scen_2000_120_sine_IP_L,146276,146276,0,offline_arrival_slot,44.301113,0.051000,2.491540,215839.952888,0,20,3.698213,0,0.000000,387512.144628,T30_sine_IP_L_offline_0_1115_2250_20/scen_2000_120_sine_IP_L/rust_offline/strategy_arrival_slot
2,T30,scen_2000_120_sine_IP_L,146276,146276,8,online_arrival_slot,44.518090,3.926000,2.504666,214999.138615,1,20,0.000988,20,13.428000,387512.144628,T30_sine_IP_L_perrequest_0_1115_2248_20/scen_2000_120_sine_IP_L/rust_online/online_arrival_slot
3,T30,scen_2000_120_sine_IP_L,146276,140685,8,online_bandit,49.138611,3.893000,3.802786,191543.455132,0,20,0.036827,20,13.462271,376598.947172,T30_sine_IP_L_perrequest_0_1115_2248_20/scen_2000_120_sine_IP_L/rust_online/online_bandit
4,T30,scen_2000_120_sine_IP_L,146276,140042,0,offline_random_slot,51.796801,0.055000,2.501625,180008.044842,0,20,4.597615,0,0.000000,373435.886444,T30_sine_IP_L_offline_0_1115_2250_20/scen_2000_120_sine_IP_L/rust_offline/strategy_random_slot
